# Exact-set and count-based analysis of mouse TRA/TRB repertoires

This notebook is the restored primary analysis for the BALB/c to C57BL/6
allotransplantation study. It retains the complete pre-refactor notebook workflow and
uses edgeR with a Fisher sensitivity analysis. Each execution is restricted to one
prespecified biological stratum.

Historical source: commit `6678766a0e8913ec2feacb3efb2daebd6a27eda4`, notebook
blob `043ec6bf8227b72ea3e3bfc6683aaac1622fe341`.

Two complementary evidence streams are evaluated:

1. exact `aaV` clonotype sets, defined by `(CDR3 amino-acid sequence, V segment)`, with
   group intersections and V-segment localization;
2. count-based inference using sample-resolved edgeR and an independent pooled-count
   Fisher test for the g1 versus g5+g6 contrast.

The notebook also retains TRA/TRB repertoire similarity, mouse-level UMI correlation,
and within-group overlap analyses from the verified historical implementation. All
figures shown during execution are persisted under one stratum-specific directory.

Treatment groups follow the study metadata: g1, intact BALB/c recipients; g2, C57BL/6
donors; g3, syngeneic bone-marrow transplantation; g4, conditioning control; g5,
allogeneic bone-marrow transplantation; and g6, allogeneic bone-marrow transplantation
with donor thymus.

The source notebook contains no inferred biological conclusion. Each executed notebook
derives its own stratum-specific ranking and explicitly distinguishes concordant,
single-method, and descriptive evidence.


## Runtime parameters and output contract


In [ ]:
from pathlib import Path
import ast
import itertools
import json
import math
import os
import re
import sys

REPOSITORY_ROOT = Path(os.environ.get("MICE_TCR_REPO", Path.cwd())).expanduser().resolve()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.runtime import configure_repseq_parallelism, configure_runtime

N_WORKERS = configure_runtime()
ACTIVE_CHAIN = "TCR"

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib_venn import venn2, venn3
import numpy as np
import pandas as pd
import seaborn as sns
from adjustText import adjust_text
from IPython.display import display

from repseq import clonosets as cl
from repseq import clustering
from repseq import diffexp
from repseq import intersections
from repseq import io as repseqio
from repseq import logo
from repseq import mixcr as mx
from repseq import slurm
from repseq import stats
from repseq import vdjtools
from repseq import clone_filter as clf

configure_repseq_parallelism(N_WORKERS)

STRATUM_CONFIG = {
    "cd4_thymus": {
        "label": "CD4 T cells, thymus",
        "cell_subsets": ("cd4",),
        "tissues": ("thymus",),
        "pool_within_mouse": False,
    },
    "cd8_thymus": {
        "label": "CD8 T cells, thymus",
        "cell_subsets": ("cd8",),
        "tissues": ("thymus",),
        "pool_within_mouse": False,
    },
    "cd4_spleen": {
        "label": "CD4 T cells, spleen",
        "cell_subsets": ("cd4",),
        "tissues": ("spleen",),
        "pool_within_mouse": False,
    },
    "cd8_spleen": {
        "label": "CD8 T cells, spleen",
        "cell_subsets": ("cd8",),
        "tissues": ("spleen",),
        "pool_within_mouse": False,
    },
    "cd4_combined": {
        "label": "CD4 T cells, thymus and spleen pooled within mouse",
        "cell_subsets": ("cd4",),
        "tissues": ("thymus", "spleen"),
        "pool_within_mouse": True,
    },
    "cd8_combined": {
        "label": "CD8 T cells, thymus and spleen pooled within mouse",
        "cell_subsets": ("cd8",),
        "tissues": ("thymus", "spleen"),
        "pool_within_mouse": True,
    },
    "thymus_combined": {
        "label": "Thymus, CD4 and CD8 pooled within mouse",
        "cell_subsets": ("cd4", "cd8"),
        "tissues": ("thymus",),
        "pool_within_mouse": True,
    },
    "spleen_combined": {
        "label": "Spleen, CD4 and CD8 pooled within mouse",
        "cell_subsets": ("cd4", "cd8"),
        "tissues": ("spleen",),
        "pool_within_mouse": True,
    },
}

STRATUM = os.environ.get("MICE_TCR_STRATUM", "cd4_thymus").strip().lower()
if STRATUM not in STRATUM_CONFIG:
    raise ValueError(
        f"Unknown MICE_TCR_STRATUM={STRATUM!r}. "
        f"Expected one of: {', '.join(STRATUM_CONFIG)}"
    )
STRATUM_DEFINITION = STRATUM_CONFIG[STRATUM]
STRATUM_LABEL = STRATUM_DEFINITION["label"]
POOL_WITHIN_MOUSE = STRATUM_DEFINITION["pool_within_mouse"]

APPROACH = "01_set_count"
FIGURE_ROOT = REPOSITORY_ROOT / "figures" / APPROACH
FIGURE_DIR = FIGURE_ROOT / STRATUM
RESULT_ROOT = REPOSITORY_ROOT / "results" / APPROACH
RESULT_DIR = RESULT_ROOT / STRATUM
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Biological stratum: {STRATUM} | {STRATUM_LABEL}")
print(f"Mouse-level pooling for count inference: {POOL_WITHIN_MOUSE}")
print(f"Figure directory: {FIGURE_DIR}")
print(f"Result directory: {RESULT_DIR}")


In [ ]:
_ORIGINAL_PLT_SAVEFIG = plt.savefig
_ORIGINAL_PLT_SHOW = plt.show
_PLOT_SEQUENCE = 0
PLOT_RECORDS = []


def _slugify_figure_name(value):
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value[:100] or "untitled_figure"


def _figure_title(fig):
    if fig._suptitle is not None and fig._suptitle.get_text().strip():
        return fig._suptitle.get_text().strip()
    titles = [axis.get_title().strip() for axis in fig.axes if axis.get_title().strip()]
    return "; ".join(titles) or "TCR repertoire analysis"


def _central_figure_path(requested_path, fig):
    global _PLOT_SEQUENCE
    _PLOT_SEQUENCE += 1
    if requested_path is None:
        stem = f"{_PLOT_SEQUENCE:03d}_{_slugify_figure_name(_figure_title(fig))}"
        return FIGURE_DIR / f"{stem}.png"

    requested = Path(str(requested_path))
    candidate = requested if requested.is_absolute() else REPOSITORY_ROOT / requested
    try:
        candidate.relative_to(FIGURE_ROOT)
    except ValueError:
        candidate = FIGURE_DIR / requested.name
    candidate.parent.mkdir(parents=True, exist_ok=True)
    return candidate.with_suffix(".png")


def _record_figure(fig, png_path, origin):
    pdf_path = png_path.with_suffix(".pdf")
    key = str(png_path.resolve())
    if key not in {record["png"] for record in PLOT_RECORDS}:
        PLOT_RECORDS.append(
            {
                "stratum": STRATUM,
                "title": _figure_title(fig),
                "origin": origin,
                "png": key,
                "pdf": str(pdf_path.resolve()),
            }
        )
    saved = set(getattr(fig, "_article_saved_png", set()))
    saved.add(key)
    fig._article_saved_png = saved



def _adaptive_annotation_color(image, value):
    red, green, blue, _ = image.cmap(image.norm(float(value)))
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return "white" if luminance < 0.52 else "black"


def _apply_article_figure_context(fig, requested_path=None):
    if getattr(fig, "_article_context_applied", False):
        return
    existing_titles = [
        axis.get_title().strip() for axis in fig.axes if axis.get_title().strip()
    ]
    existing_suptitle = (
        fig._suptitle.get_text().strip() if fig._suptitle is not None else ""
    )
    explicit_text = " ".join(
        [existing_suptitle, *existing_titles, str(requested_path or "")]
    )
    upper = explicit_text.upper()
    if "TRA" in upper and "TRB" in upper:
        chain_context = "TRA and TRB"
    elif "TRB" in upper:
        chain_context = "TRB"
    elif "TRA" in upper or "TRAV" in upper or "AAV" in upper:
        chain_context = "TRA"
    else:
        chain_context = ACTIVE_CHAIN

    groups = sorted(set(re.findall(r"(?<![A-Za-z0-9])g[1-6](?![A-Za-z0-9])", explicit_text.lower())))
    group_context = (
        "treatment groups " + ", ".join(groups)
        if groups
        else "stratum-level treatment comparison"
    )
    context = f"{chain_context} | {group_context} | {STRATUM_LABEL}"
    combined_title = " ".join([existing_suptitle, *existing_titles])
    if STRATUM_LABEL.lower() not in combined_title.lower():
        if existing_suptitle:
            fig._suptitle.set_text(context + "\n" + existing_suptitle)
        else:
            fig.suptitle(context, fontsize=13, y=1.02)
    fig._article_context_applied = True

def _save_figure_pair(fig, requested_path=None, origin="inline", **kwargs):
    _apply_article_figure_context(fig, requested_path=requested_path)
    png_path = _central_figure_path(requested_path, fig)
    pdf_path = png_path.with_suffix(".pdf")
    png_options = dict(kwargs)
    png_options.pop("format", None)
    png_options.setdefault("dpi", 300)
    png_options.setdefault("bbox_inches", "tight")
    fig.savefig(png_path, **png_options)
    pdf_options = dict(png_options)
    pdf_options.pop("dpi", None)
    fig.savefig(pdf_path, **pdf_options)
    _record_figure(fig, png_path, origin)
    return png_path


def _article_savefig(path, *args, **kwargs):
    if args:
        raise TypeError("Positional savefig arguments are not supported by this notebook.")
    saved_path = _save_figure_pair(plt.gcf(), path, origin="explicit", **kwargs)
    print(f"Saved figure: {saved_path.relative_to(REPOSITORY_ROOT)}")
    return None


def _persist_open_figures(origin="inline"):
    for figure_number in list(plt.get_fignums()):
        fig = plt.figure(figure_number)
        if not getattr(fig, "_article_saved_png", set()):
            _save_figure_pair(fig, origin=origin)


def _article_show(*args, **kwargs):
    _persist_open_figures(origin="displayed")
    return _ORIGINAL_PLT_SHOW(*args, **kwargs)


plt.savefig = _article_savefig
plt.show = _article_show

sns.set_theme(style="white", context="notebook")
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "savefig.facecolor": "white",
    }
)


In [ ]:
WORKING_DIR = Path(
    os.environ.get("MICE_TCR_WORKING_DIR", "/projects/mice_transplant_2025/test_run")
).expanduser().resolve()
METADATA_FILENAME = Path(
    os.environ.get(
        "MICE_TCR_METADATA_CSV",
        "/projects/mice_transplant_2025/metadata_mice_transplant.csv",
    )
).expanduser().resolve()
CLONOSET_INDEX_FILENAME = Path(
    os.environ.get(
        "MICE_TCR_CLONOSET_INDEX",
        WORKING_DIR / "clonosets_mice_transplant_2025_df.csv",
    )
).expanduser().resolve()

for required_path in (METADATA_FILENAME, CLONOSET_INDEX_FILENAME):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required input file not found: {required_path}")

metadata_all = pd.read_csv(METADATA_FILENAME)
required_metadata_columns = {
    "chain", "sample_no", "sample_id", "group_no", "mouse_no", "source", "subtype"
}
missing_metadata = required_metadata_columns.difference(metadata_all.columns)
if missing_metadata:
    raise ValueError(f"Metadata columns are missing: {sorted(missing_metadata)}")


def _identifier(value):
    if pd.isna(value):
        return ""
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value).strip()
    return str(int(numeric)) if numeric.is_integer() else str(value).strip()


metadata_all = metadata_all.copy()
metadata_all["sample_id_old"] = (
    metadata_all["chain"].astype(str).str.strip()
    + "-"
    + metadata_all["sample_no"].map(_identifier)
)
metadata_all = metadata_all.drop(columns="chain")

clonosets = pd.read_csv(CLONOSET_INDEX_FILENAME)
required_index_columns = {"sample_id", "filename", "chain"}
missing_index = required_index_columns.difference(clonosets.columns)
if missing_index:
    raise ValueError(f"Clonoset-index columns are missing: {sorted(missing_index)}")

clonosets = clonosets.rename(columns={"sample_id": "sample_id_old"})
clonosets = clonosets.merge(
    metadata_all,
    on="sample_id_old",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched = clonosets.loc[clonosets["_merge"].ne("both"), "sample_id_old"].tolist()
if unmatched:
    raise ValueError(
        "Clonoset rows could not be matched to metadata by the historical "
        "chain/sample-number key: " + ", ".join(map(str, unmatched[:10]))
    )
clonosets = clonosets.drop(columns=["sample_id_old", "_merge"])


def _resolve_clonoset_path(value):
    path = Path(str(value)).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend(
            [CLONOSET_INDEX_FILENAME.parent / path, WORKING_DIR / "mixcr" / path]
        )
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    return str(candidates[-1].resolve())


clonosets["filename"] = clonosets["filename"].map(_resolve_clonoset_path)
missing_clonosets = [path for path in clonosets["filename"] if not Path(path).is_file()]
if missing_clonosets:
    raise FileNotFoundError(
        "MiXCR clonotype exports referenced by the index were not found: "
        + "; ".join(missing_clonosets[:10])
    )

classification_text = (
    clonosets["subtype"].fillna("").astype(str).str.lower()
    + " "
    + clonosets["sample_id"].fillna("").astype(str).str.lower()
)
source_text = (
    clonosets["source"].fillna("").astype(str).str.lower()
    + " "
    + clonosets["sample_id"].fillna("").astype(str).str.lower()
)
clonosets["_cell_subset"] = np.select(
    [
        classification_text.str.contains(r"(?:^|[^a-z0-9])cd4(?:[^a-z0-9]|$)", regex=True),
        classification_text.str.contains(r"(?:^|[^a-z0-9])cd8(?:[^a-z0-9]|$)", regex=True),
    ],
    ["cd4", "cd8"],
    default="unresolved",
)
clonosets["_tissue"] = np.select(
    [source_text.str.contains("spleen"), source_text.str.contains("thym")],
    ["spleen", "thymus"],
    default="unresolved",
)
unresolved = clonosets[
    clonosets["_cell_subset"].eq("unresolved") | clonosets["_tissue"].eq("unresolved")
]["sample_id"].unique()
if len(unresolved):
    raise ValueError(
        "Every sample must map unambiguously to CD4/CD8 and thymus/spleen. "
        f"Unresolved samples: {', '.join(map(str, unresolved[:10]))}"
    )

clonosets = clonosets[
    clonosets["_cell_subset"].isin(STRATUM_DEFINITION["cell_subsets"])
    & clonosets["_tissue"].isin(STRATUM_DEFINITION["tissues"])
].copy()
if clonosets.empty:
    raise ValueError(f"No samples remain for biological stratum {STRATUM!r}.")

clonosets["chain"] = clonosets["chain"].astype(str).str.upper().str.strip()
clonosets["group_no"] = clonosets["group_no"].astype(str).str.lower().str.strip()
clonosets = clonosets[
    ["sample_id"] + [column for column in clonosets.columns if column != "sample_id"]
].sort_values("sample_id").reset_index(drop=True)

metadata = clonosets.drop_duplicates("sample_id").copy()
metadata["mouse_id"] = (
    metadata["group_no"].astype(str)
    + "_m"
    + metadata["mouse_no"].map(_identifier)
)

historical_exclusions = {
    "g1_m3_thymus_cd8_80_alpha",
    "g5_m1_spleen_cd4_93_alpha",
    "g5_m4_thymus_cd8_100_alpha",
}

def _group_samples(chain, group_no):
    selected = clonosets[
        clonosets["chain"].eq(chain) & clonosets["group_no"].eq(group_no)
    ].copy()
    if chain == "TRA":
        selected = selected[~selected["sample_id"].isin(historical_exclusions)]
    return selected.sort_values("umi_func")


for chain_name in ("TRA", "TRB"):
    for group_name in ("g1", "g2", "g3", "g4", "g5", "g6"):
        globals()[f"clonosets_{chain_name.lower()}_{group_name}"] = _group_samples(
            chain_name, group_name
        )

clonosets_tra_g6_allo = clonosets_tra_g6[
    ~clonosets_tra_g6["source"].astype(str).str.lower().eq("thymus")
].copy()
clonosets_tra_g6_auto = clonosets_tra_g6[
    ~clonosets_tra_g6["source"].astype(str).str.lower().eq("allothymus")
].copy()
clonosets_trb_g6_allo = clonosets_trb_g6[
    ~clonosets_trb_g6["source"].astype(str).str.lower().eq("thymus")
].copy()
clonosets_trb_g6_auto = clonosets_trb_g6[
    ~clonosets_trb_g6["source"].astype(str).str.lower().eq("allothymus")
].copy()

_REPSEQ_COUNT_TABLE = intersections.count_table


def _safe_count_table(sample_index, *args, **kwargs):
    if sample_index is None or len(sample_index) == 0:
        print("No samples were available for this group; returning an empty count table.")
        return pd.DataFrame(index=pd.Index([], dtype="object"))
    return _REPSEQ_COUNT_TABLE(sample_index, *args, **kwargs)


intersections.count_table = _safe_count_table

sample_audit = (
    metadata.groupby(["chain", "group_no"], observed=True)
    .agg(samples=("sample_id", "nunique"), mice=("mouse_id", "nunique"))
    .reset_index()
)
sample_audit.to_csv(RESULT_DIR / "selected_samples_by_chain_and_group.csv", index=False)
display(sample_audit)


## TRA exact-set analysis


In [ ]:
ACTIVE_CHAIN = "TRA"
ct_g1_aaV_tra = intersections.count_table(
    clonosets_tra_g1,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g2_aaV_tra = intersections.count_table(
    clonosets_tra_g2,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g3_aaV_tra = intersections.count_table(
    clonosets_tra_g3,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g4_aaV_tra = intersections.count_table(
    clonosets_tra_g4,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g5_aaV_tra = intersections.count_table(
    clonosets_tra_g5,
    
    overlap_type="aaV",
    mismatches=0
)













ct_g6_aaV_tra = intersections.count_table(
    clonosets_tra_g6,
    
    overlap_type="aaV",
    mismatches=0
)

In [ ]:
def plot_double_triple_venn_like(
    center_set,
    left_top_set,
    left_bottom_set,
    right_top_set,
    right_bottom_set,
    center_label,
    left_top_label,
    left_bottom_label,
    right_top_label,
    right_bottom_label,
    title=None,
    save_path=None
):
    """
    5-circle venn-like plot:

    The left triplet: left_top, center, left_bottom
    The right triplet: right_top, center, right_bottom

    Cross-intersections between the left and right triplets
    are not displayed.
    """

    center_set = set(center_set)
    left_top_set = set(left_top_set)
    left_bottom_set = set(left_bottom_set)
    right_top_set = set(right_top_set)
    right_bottom_set = set(right_bottom_set)

    
    
    
    left_top_only = left_top_set - (center_set | left_bottom_set)
    left_bottom_only = left_bottom_set - (center_set | left_top_set)
    left_top_left_bottom_only = (left_top_set & left_bottom_set) - center_set
    center_left_top_only = (center_set & left_top_set) - left_bottom_set
    center_left_bottom_only = (center_set & left_bottom_set) - left_top_set
    left_triple = center_set & left_top_set & left_bottom_set

    
    
    
    right_top_only = right_top_set - (center_set | right_bottom_set)
    right_bottom_only = right_bottom_set - (center_set | right_top_set)
    right_top_right_bottom_only = (right_top_set & right_bottom_set) - center_set
    center_right_top_only = (center_set & right_top_set) - right_bottom_set
    center_right_bottom_only = (center_set & right_bottom_set) - right_top_set
    right_triple = center_set & right_top_set & right_bottom_set

    
    
    
    center_only_after_all = center_set - (
        left_top_set | left_bottom_set | right_top_set | right_bottom_set
    )

    
    
    
    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.set_aspect("equal")
    ax.axis("off")

    colors = {
        left_top_label: "#e41a1c",
        center_label: "#4daf4a",
        left_bottom_label: "#377eb8",
        right_top_label: "#ff7f00",
        right_bottom_label: "#984ea3",
    }

    circles = {
        left_top_label: {
            "xy": (-2.0, 1.0),
            "r": 1.35,
            "set": left_top_set
        },
        left_bottom_label: {
            "xy": (-2.0, -1.0),
            "r": 1.35,
            "set": left_bottom_set
        },
        center_label: {
            "xy": (0.0, 0.0),
            "r": 1.55,
            "set": center_set
        },
        right_top_label: {
            "xy": (2.0, 1.0),
            "r": 1.35,
            "set": right_top_set
        },
        right_bottom_label: {
            "xy": (2.0, -1.0),
            "r": 1.35,
            "set": right_bottom_set
        }
    }

    for label, params in circles.items():
        c = Circle(
            params["xy"],
            params["r"],
            facecolor=colors[label],
            edgecolor="black",
            linewidth=2,
            alpha=0.28,
            zorder=1
        )
        ax.add_patch(c)

    label_positions = {
        left_top_label: (-3.0, 1.65),
        left_bottom_label: (-3.0, -1.65),
        center_label: (0.0, -1.55),
        right_top_label: (3.0, 1.65),
        right_bottom_label: (3.0, -1.65),
    }

    for label, (x, y) in label_positions.items():
        n = len(circles[label]["set"])
        ax.text(
            x, y,
            f"{label}\n{n}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            zorder=5
        )

    def put_count(x, y, value):
        ax.text(
            x, y,
            f"{len(value)}",
            ha="center",
            va="center",
            fontsize=12,
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.18",
                facecolor="white",
                edgecolor="none",
                alpha=0.75
            ),
            zorder=10
        )

    
    put_count(-2.55, 1.05, left_top_only)
    put_count(-2.55, -1.05, left_bottom_only)
    put_count(-2.05, 0.00, left_top_left_bottom_only)
    put_count(-1.05, 0.62, center_left_top_only)
    put_count(-1.05, -0.62, center_left_bottom_only)

    
    put_count(-1.30, 0.00, left_triple)
    put_count(0.00, 0.00, center_only_after_all)
    put_count(1.30, 0.00, right_triple)

    
    put_count(2.55, 1.05, right_top_only)
    put_count(2.55, -1.05, right_bottom_only)
    put_count(2.05, 0.00, right_top_right_bottom_only)
    put_count(1.05, 0.62, center_right_top_only)
    put_count(1.05, -0.62, center_right_bottom_only)

    if title is not None:
        ax.set_title(title, fontsize=15, pad=20)

    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-2.5, 2.5)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return {
        "center_only_after_all": center_only_after_all,
        "left_triple": left_triple,
        "right_triple": right_triple,
        "left_regions": {
            "left_top_only": left_top_only,
            "left_bottom_only": left_bottom_only,
            "left_top_left_bottom_only": left_top_left_bottom_only,
            "center_left_top_only": center_left_top_only,
            "center_left_bottom_only": center_left_bottom_only,
            "left_triple": left_triple,
        },
        "right_regions": {
            "right_top_only": right_top_only,
            "right_bottom_only": right_bottom_only,
            "right_top_right_bottom_only": right_top_right_bottom_only,
            "center_right_top_only": center_right_top_only,
            "center_right_bottom_only": center_right_bottom_only,
            "right_triple": right_triple,
        }
    }


g1_tra_aaV = set(ct_g1_aaV_tra.index)
g2_tra_aaV = set(ct_g2_aaV_tra.index)
g3_tra_aaV = set(ct_g3_aaV_tra.index)
g4_tra_aaV = set(ct_g4_aaV_tra.index)
g5_tra_aaV = set(ct_g5_aaV_tra.index)
g6_tra_aaV = set(ct_g6_aaV_tra.index)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)


res_g1_tra = plot_double_triple_venn_like(
    center_set=g1_tra_aaV,
    left_top_set=g5_tra_aaV,
    left_bottom_set=g6_tra_aaV,
    right_top_set=g2_tra_aaV,
    right_bottom_set=g4_tra_aaV,
    center_label="g1",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g2",
    right_bottom_label="g4",
    title="TRA clonotypes overlap: g5-g1-g6 and g2-g1-g4, aaV",
    save_path=str(FIGURE_DIR / "TRA_aaV_double_triple_g1.png")
)


res_g2_tra = plot_double_triple_venn_like(
    center_set=g2_tra_aaV,
    left_top_set=g5_tra_aaV,
    left_bottom_set=g6_tra_aaV,
    right_top_set=g1_tra_aaV,
    right_bottom_set=g3_tra_aaV,
    center_label="g2",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g1",
    right_bottom_label="g3",
    title="TRA clonotypes overlap: g5-g2-g6 and g1-g2-g3, aaV",
    save_path=str(FIGURE_DIR / "TRA_aaV_double_triple_g2.png")
)

In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    clone_set = sorted(clone_set)

    if overlap_type == "aaV":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def plot_v_distribution(df, title, top_n=20):
    if df.empty:
        print(f"{title}: empty dataframe")
        return

    vc = df["v"].value_counts()

    if top_n is not None:
        vc = vc.head(top_n)

    plt.figure(figsize=(12, 6))
    vc.plot(kind="bar")
    plt.title(title, fontsize=14)
    plt.xlabel("V gene", fontsize=12)
    plt.ylabel("Number of unique clonotypes", fontsize=12)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
    

g1 = set(ct_g1_aaV_tra.index)
g2 = set(ct_g2_aaV_tra.index)
g3 = set(ct_g3_aaV_tra.index)
g4 = set(ct_g4_aaV_tra.index)
g5 = set(ct_g5_aaV_tra.index)
g6 = set(ct_g6_aaV_tra.index)




g1_remaining_aaV_tra = g1 - (g2 | g4 | g5 | g6)




g2_remaining_aaV_tra = g2 - (g1 | g3 | g5 | g6)



clones_g1_remaining_aaV_tra_df = clone_set_to_df(
    g1_remaining_aaV_tra,
    overlap_type="aaV"
)

clones_g2_remaining_aaV_tra_df = clone_set_to_df(
    g2_remaining_aaV_tra,
    overlap_type="aaV"
)

plot_v_distribution(
    clones_g1_remaining_aaV_tra_df,
    "TRA: V genes among clonotypes remaining in g1 after subtracting g2, g4, g5, g6, aaV"
)

plot_v_distribution(
    clones_g2_remaining_aaV_tra_df,
    "TRA: V genes among clonotypes remaining in g2 after subtracting g1, g3, g5, g6, aaV"
)

In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    if overlap_type == "aaV":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def make_v_summary_from_clone_set(clone_set, total_n, overlap_type="aaV"):
    df = clone_set_to_df(clone_set, overlap_type=overlap_type)

    if df.empty:
        return pd.DataFrame(columns=["v", "n_cdr3", "frequency", "frequency_pct"])

    out = (
        df["v"]
        .value_counts()
        .rename_axis("v")
        .reset_index(name="n_cdr3")
    )

    out["frequency"] = out["n_cdr3"] / total_n
    out["frequency_pct"] = 100 * out["frequency"]

    return out


def compare_full_vs_remaining_after_subtractions(
    target_ct,
    subtract_cts,
    target_label,
    overlap_type="aaV"
):
    target_set = set(target_ct.index)
    subtract_union = set().union(*[set(ct.index) for ct in subtract_cts])

    remaining_set = target_set - subtract_union

    full_v = make_v_summary_from_clone_set(
        target_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    remaining_v = make_v_summary_from_clone_set(
        remaining_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    full_v = full_v.rename(columns={
        "n_cdr3": f"cdr3_in_full_{target_label}",
        "frequency": f"frequency_in_full_{target_label}",
        "frequency_pct": f"frequency_in_full_{target_label}_pct"
    })

    remaining_v = remaining_v.rename(columns={
        "n_cdr3": f"cdr3_remaining_in_{target_label}",
        "frequency": f"frequency_remaining_in_{target_label}_relative_to_original",
        "frequency_pct": f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    })

    cmp_df = full_v.merge(remaining_v, on="v", how="left")

    remaining_count_col = f"cdr3_remaining_in_{target_label}"
    full_count_col = f"cdr3_in_full_{target_label}"

    remaining_freq_col = f"frequency_remaining_in_{target_label}_relative_to_original"
    remaining_freq_pct_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_freq_pct_col = f"frequency_in_full_{target_label}_pct"

    cmp_df[remaining_count_col] = cmp_df[remaining_count_col].fillna(0).astype(int)
    cmp_df[remaining_freq_col] = cmp_df[remaining_freq_col].fillna(0.0)
    cmp_df[remaining_freq_pct_col] = cmp_df[remaining_freq_pct_col].fillna(0.0)

    cmp_df["lost_cdr3"] = cmp_df[full_count_col] - cmp_df[remaining_count_col]

    cmp_df["frequency_change_pct"] = (
        cmp_df[full_freq_pct_col] - cmp_df[remaining_freq_pct_col]
    )

    cmp_df["retained_cdr3_share"] = (
        cmp_df[remaining_count_col] / cmp_df[full_count_col]
    )

    cmp_df["retained_cdr3_share_pct"] = (
        100 * cmp_df["retained_cdr3_share"]
    )

    cmp_df["loss_share"] = cmp_df["lost_cdr3"] / cmp_df[full_count_col]
    cmp_df["loss_share_pct"] = 100 * cmp_df["loss_share"]

    cmp_df_changed = (
        cmp_df[cmp_df["lost_cdr3"] != 0]
        .sort_values(
            ["loss_share", "lost_cdr3", full_count_col],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )

    return cmp_df, cmp_df_changed, remaining_set


cmp_g1_tra_all, cmp_g1_tra_changed_ranked, g1_remaining_aaV_tra = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g1_aaV_tra,
        subtract_cts=[
            ct_g2_aaV_tra,
            ct_g4_aaV_tra,
            ct_g5_aaV_tra,
            ct_g6_aaV_tra
        ],
        target_label="g1",
        overlap_type="aaV"
    )
)

display(cmp_g1_tra_changed_ranked.head(20))

cmp_g1_tra_changed_ranked.to_csv(
    RESULT_DIR / "TRA_cmp_g1_remaining_after_g2_g4_g5_g6_ranked.csv",
    index=False
)


cmp_g2_tra_all, cmp_g2_tra_changed_ranked, g2_remaining_aaV_tra = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g2_aaV_tra,
        subtract_cts=[
            ct_g1_aaV_tra,
            ct_g3_aaV_tra,
            ct_g5_aaV_tra,
            ct_g6_aaV_tra
        ],
        target_label="g2",
        overlap_type="aaV"
    )
)

display(cmp_g2_tra_changed_ranked.head(20))

cmp_g2_tra_changed_ranked.to_csv(
    RESULT_DIR / "TRA_cmp_g2_remaining_after_g1_g3_g5_g6_ranked.csv",
    index=False
)

In [ ]:
def make_clone_intersection_table_venn3(
    ct_a,
    ct_b,
    ct_c,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
):
    set_a = set(ct_a.index)
    set_b = set(ct_b.index)
    set_c = set(ct_c.index)

    all_clones = set_a | set_b | set_c

    sum_a = ct_a.sum(axis=1).astype(float).to_dict()
    sum_b = ct_b.sum(axis=1).astype(float).to_dict()
    sum_c = ct_c.sum(axis=1).astype(float).to_dict()

    total_a = float(ct_a.sum(axis=1).sum())
    total_b = float(ct_b.sum(axis=1).sum())
    total_c = float(ct_c.sum(axis=1).sum())

    region_names = {
        "100": f"{label_a} only",
        "010": f"{label_b} only",
        "001": f"{label_c} only",
        "110": f"({label_a}∩{label_b})-{label_c}",
        "101": f"({label_a}∩{label_c})-{label_b}",
        "011": f"({label_b}∩{label_c})-{label_a}",
        "111": f"{label_a}∩{label_b}∩{label_c}",
    }

    rows = []

    for clone in sorted(all_clones):
        in_a = clone in set_a
        in_b = clone in set_b
        in_c = clone in set_c

        region_id = (
            ("1" if in_a else "0") +
            ("1" if in_b else "0") +
            ("1" if in_c else "0")
        )

        if overlap_type == "aaV":
            cdr3aa, v = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "clone_id": clone,
            }

        elif overlap_type == "aaVJ":
            cdr3aa, v, j = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "j": j,
                "clone_id": clone,
            }

        else:
            raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")

        row_sum_a = float(sum_a.get(clone, 0.0))
        row_sum_b = float(sum_b.get(clone, 0.0))
        row_sum_c = float(sum_c.get(clone, 0.0))

        row.update({
            f"in_{label_a}": in_a,
            f"in_{label_b}": in_b,
            f"in_{label_c}": in_c,

            "region_id": region_id,
            "region_name": region_names[region_id],

            f"row_sum_{label_a}": row_sum_a,
            f"row_sum_{label_b}": row_sum_b,
            f"row_sum_{label_c}": row_sum_c,

            f"freq_vs_{label_a}": row_sum_a / total_a if total_a > 0 else np.nan,
            f"freq_vs_{label_b}": row_sum_b / total_b if total_b > 0 else np.nan,
            f"freq_vs_{label_c}": row_sum_c / total_c if total_c > 0 else np.nan,
        })

        row[f"freq_vs_{label_a}_pct"] = 100 * row[f"freq_vs_{label_a}"]
        row[f"freq_vs_{label_b}_pct"] = 100 * row[f"freq_vs_{label_b}"]
        row[f"freq_vs_{label_c}_pct"] = 100 * row[f"freq_vs_{label_c}"]

        rows.append(row)

    out = (
        pd.DataFrame(rows)
        .sort_values(
            ["region_id", f"row_sum_{label_a}", f"row_sum_{label_b}", f"row_sum_{label_c}", "v", "cdr3aa"],
            ascending=[True, False, False, False, True, True]
        )
        .reset_index(drop=True)
    )

    return out

g1_g5_g6_intersection_table = make_clone_intersection_table_venn3(
    ct_a=ct_g1_aaV_tra,
    ct_b=ct_g5_aaV_tra,
    ct_c=ct_g6_aaV_tra,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
)

display(g1_g5_g6_intersection_table.head(20))

In [ ]:
def make_v_region_summary(intersection_df, freq_col):
    out = (
        intersection_df
        .groupby(["v", "region_id", "region_name"], as_index=False)
        .agg(
            n_clones=("clone_id", "nunique"),
            burden=(freq_col, "sum"),
            median_clone_freq=(freq_col, "median"),
            mean_clone_freq=(freq_col, "mean")
        )
    )

    out["region_share_within_v"] = (
        out["burden"] /
        out.groupby("v")["burden"].transform("sum")
    )

    return out

def make_v_profile(cmp_df, intersection_df, target_label, freq_col):
    region_summary = make_v_region_summary(intersection_df, freq_col=freq_col)

    share_wide = (
        region_summary
        .pivot(index="v", columns="region_id", values="region_share_within_v")
        .fillna(0)
    )

    
    p = share_wide.replace(0, np.nan)
    entropy = -(p * np.log2(p)).sum(axis=1).fillna(0) / np.log2(7)
    share_wide["region_entropy"] = entropy

    profile = cmp_df.merge(
        share_wide.reset_index(),
        on="v",
        how="left"
    ).fillna(0)

    
    profile["private_share"] = profile.get("100", 0)
    profile["triple_share"] = profile.get("111", 0)

    full_freq_col = f"frequency_in_full_{target_label}_pct"
    full_count_col = f"cdr3_in_full_{target_label}"

    profile["impact_score"] = (
        profile[full_freq_col] *
        profile["loss_share"]
    )

    profile["specific_score"] = (
        profile[full_freq_col] *
        profile["retained_cdr3_share"]
    )

    profile = profile.sort_values("impact_score", ascending=False).reset_index(drop=True)

    return profile, region_summary

g1_profile, g1_v_region_summary = make_v_profile(
    cmp_df=cmp_g1_tra_all,                      
    intersection_df=g1_g5_g6_intersection_table,
    target_label="g1",
    freq_col="freq_vs_g1"
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

plt.rcParams["font.family"] = "DejaVu Sans"

REGION_LABELS = {
    "100": "target group only",
    "110": "target group ∩ g5",
    "101": "target group ∩ g6",
    "111": "target group ∩ g5 ∩ g6",
    "010": "g5 only",
    "011": "g5 ∩ g6",
    "001": "g6 only",
}

METRIC_LABELS = {
    "private_share": "share of target-group-only clonotypes",
    "triple_share": "share of clonotypes in the triple intersection",
    "region_entropy": "normalized entropy across overlap regions",
    "impact_score": "loss index",
    "specific_score": "specificity index",
}


def plot_v_dumbbell(profile, target_label, top_n=20, min_full_freq_pct=0.5):
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    remaining_freq_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_count_col = f"cdr3_in_full_{target_label}"
    remaining_count_col = f"cdr3_remaining_in_{target_label}"

    plot_df = profile.copy()

    
    plot_df = plot_df[plot_df[full_freq_col] >= min_full_freq_pct].copy()

    if plot_df.empty:
        print(
            f"No V segments with an initial share >= {min_full_freq_pct}% "
            f"for group {target_label}"
        )
        return pd.DataFrame()

    
    plot_df["count_loss"] = (
        plot_df[full_count_col] - plot_df[remaining_count_col]
    )

    
    
    plot_df["scaled_retention_score"] = (
        plot_df[full_count_col] / (1 + plot_df["count_loss"])
    )

    
    plot_df = (
        plot_df
        .sort_values(
            ["scaled_retention_score", full_count_col, remaining_count_col],
            ascending=[False, False, False]
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    y = np.arange(len(plot_df))

    plt.figure(figsize=(10, max(8, 0.35 * len(plot_df))))

    for i, row in plot_df.iterrows():
        plt.plot(
            [row[remaining_freq_col], row[full_freq_col]],
            [i, i],
            linewidth=2,
            alpha=0.75
        )

    plt.scatter(
        plot_df[remaining_freq_col],
        y,
        s=65,
        label="retained after all subtractions"
    )

    plt.scatter(
        plot_df[full_freq_col],
        y,
        s=65,
        label="complete group"
    )

    plt.yticks(y, plot_df["v"])

    
    plt.gca().invert_yaxis()

    plt.xlabel("Clonotypes assigned to the V segment, %")
    plt.ylabel("V segment")
    plt.title(
        f"TRA, {target_label}: V segments with an initial share ≥ {min_full_freq_pct}% "
        f"and the highest retention score"
    )

    plt.legend(title="Set state")
    plt.tight_layout()
    plt.show()

    return plot_df


def get_bubble_labeled_v_genes(profile, target_label, top_n=10):
    full_count_col = f"cdr3_in_full_{target_label}"
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    required = {
        "v", "impact_score", "retained_cdr3_share_pct",
        full_count_col, full_freq_col,
    }
    missing = sorted(required.difference(profile.columns))
    if missing:
        raise ValueError(f"The V-segment profile is missing columns: {missing}")

    candidates = profile.copy()
    numeric_columns = [
        "impact_score", "retained_cdr3_share_pct", full_count_col, full_freq_col
    ]
    for column in numeric_columns:
        candidates[column] = pd.to_numeric(candidates[column], errors="coerce")
    candidates = candidates[
        candidates[full_count_col].gt(0)
        & candidates[full_freq_col].gt(0)
        & candidates["retained_cdr3_share_pct"].notna()
        & candidates["impact_score"].notna()
    ].copy()
    if candidates.empty:
        raise ValueError(
            f"No positive target-group V segments were available for {target_label}."
        )

    label_df = (
        candidates.sort_values(
            ["impact_score", "retained_cdr3_share_pct", full_count_col, "v"],
            ascending=[False, False, False, True],
        )
        .head(int(top_n))
        .reset_index(drop=True)
    )
    return label_df["v"].tolist(), label_df



def plot_v_landscape(
    profile,
    target_label,
    color_by="region_entropy",
    retained_threshold=None,
    top_n_labels=10,
):
    chain_label = "TRA"
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    full_count_col = f"cdr3_in_full_{target_label}"

    plot_df = profile.copy()
    for column in [full_freq_col, full_count_col, "retained_cdr3_share_pct", color_by]:
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")
    plot_df = plot_df[
        plot_df[full_count_col].gt(0)
        & plot_df[full_freq_col].gt(0)
        & plot_df["retained_cdr3_share_pct"].notna()
        & plot_df[color_by].notna()
    ].copy()
    if plot_df.empty:
        raise ValueError(
            f"No positive target-group V segments were available for {target_label}."
        )

    labeled_genes, label_df = get_bubble_labeled_v_genes(
        profile=plot_df,
        target_label=target_label,
        top_n=top_n_labels,
    )
    maximum_count = float(plot_df[full_count_col].max())
    marker_size = 40 + 500 * plot_df[full_count_col] / maximum_count

    fig, ax = plt.subplots(figsize=(11, 8))
    scatter = ax.scatter(
        plot_df[full_freq_col],
        plot_df["retained_cdr3_share_pct"],
        c=plot_df[color_by],
        s=marker_size,
        alpha=0.78,
    )

    y_min = float(plot_df["retained_cdr3_share_pct"].min())
    y_max = float(plot_df["retained_cdr3_share_pct"].max())
    y_span = max(y_max - y_min, 1.0)
    y_padding = max(0.08 * y_span, 2.0)
    ax.set_ylim(max(0.0, y_min - y_padding), min(105.0, y_max + y_padding))

    x_min = float(plot_df[full_freq_col].min())
    x_max = float(plot_df[full_freq_col].max())
    x_span = max(x_max - x_min, 0.1)
    x_padding = max(0.08 * x_span, 0.03)
    ax.set_xlim(max(0.0, x_min - 0.2 * x_padding), x_max + x_padding)

    if retained_threshold is not None and y_min <= retained_threshold <= y_max:
        ax.axhline(
            retained_threshold,
            linestyle=":",
            linewidth=1.3,
            color="#4C78A8",
            alpha=0.8,
        )

    texts = []
    for row in label_df.itertuples():
        texts.append(
            ax.text(
                getattr(row, full_freq_col),
                row.retained_cdr3_share_pct,
                row.v,
                fontsize=9,
                ha="left",
                va="bottom",
                bbox=dict(
                    boxstyle="round,pad=0.18",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.78,
                ),
            )
        )
    if texts:
        adjust_text(
            texts,
            x=label_df[full_freq_col].values,
            y=label_df["retained_cdr3_share_pct"].values,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.6, alpha=0.7),
            expand=(1.2, 1.4),
            force_static=0.8,
            force_text=1.0,
            iter_lim=700,
            ax=ax,
        )

    ax.set_xlabel(f"V-segment share in the complete group {target_label}, %")
    ax.set_ylabel("Retained clonotypes, %")
    ax.set_title(
        f"{chain_label} | treatment group {target_label} | {STRATUM_LABEL}\n"
        f"V-segment retention landscape; top {len(label_df)} labels by impact score"
    )
    ax.grid(False)
    fig.colorbar(
        scatter,
        ax=ax,
        pad=0.03,
        label=METRIC_LABELS.get(color_by, color_by),
    )
    fig.tight_layout()
    plt.show()
    return labeled_genes, label_df



def plot_v_region_heatmap_for_labeled_genes(
    profile,
    labeled_genes,
    target_label,
    save_path=None,
):
    chain_label = "TRA"
    region_order = ["100", "110", "101", "111", "010", "011", "001"]
    plot_df = profile.copy()
    for region in region_order:
        if region not in plot_df.columns:
            plot_df[region] = 0.0

    selected_genes = list(dict.fromkeys(labeled_genes))[:10]
    order_map = {gene: rank for rank, gene in enumerate(selected_genes)}
    plot_df = plot_df[plot_df["v"].isin(selected_genes)].copy()
    if plot_df.empty:
        raise ValueError(
            f"No bubble-selected V segments were available for the {chain_label} heatmap."
        )
    plot_df["bubble_order"] = plot_df["v"].map(order_map)
    plot_df = plot_df.sort_values("bubble_order").set_index("v")
    heatmap = plot_df[region_order].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    x_labels = [
        REGION_LABELS[region].replace("target group", target_label)
        for region in region_order
    ]

    fig_height = max(5.0, 0.48 * len(heatmap) + 2.0)
    fig, ax = plt.subplots(figsize=(14, fig_height))
    image = ax.imshow(
        heatmap.values,
        aspect="auto",
        interpolation="none",
        cmap="magma",
        vmin=0.0,
        vmax=max(1.0, float(np.nanmax(heatmap.values))),
    )
    ax.set_xticks(range(len(region_order)), labels=x_labels, rotation=35, ha="right")
    ax.set_yticks(range(len(heatmap.index)), labels=heatmap.index)
    ax.grid(False)
    ax.tick_params(which="minor", bottom=False, left=False)

    for row_index in range(heatmap.shape[0]):
        for column_index in range(heatmap.shape[1]):
            value = float(heatmap.iloc[row_index, column_index])
            ax.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=_adaptive_annotation_color(image, value),
            )

    ax.set_xlabel("Overlap region")
    ax.set_ylabel(f"{chain_label} V segment")
    ax.set_title(
        f"{chain_label} | treatment group {target_label} | {STRATUM_LABEL}\n"
        f"Overlap architecture of the top {len(heatmap)} bubble-plot V segments"
    )
    fig.colorbar(
        image,
        ax=ax,
        pad=0.03,
        label="Share of each V segment's clonotypes in the overlap region",
    )
    fig.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return plot_df.reset_index()



g1_dumbbell_selected_df = plot_v_dumbbell(
    g1_profile,
    target_label="g1",
    top_n=20,
    min_full_freq_pct=0.6
)


g1_tra_labeled_genes, g1_tra_bubble_labels_df = plot_v_landscape(
    g1_profile,
    target_label="g1",
    color_by="region_entropy",
    retained_threshold=66,
    top_n_labels=10,
)



g1_tra_heatmap_labeled_df = plot_v_region_heatmap_for_labeled_genes(
    g1_profile,
    labeled_genes=g1_tra_labeled_genes,
    target_label="g1",
    save_path=str(FIGURE_DIR / "TRA_g1_top10_bubble_genes_heatmap.png"),
)


















In [ ]:
def plot_heatmap_first_column_as_barh(
    heatmap_df,
    target_label="g1",
    value_region="100",
    top_n=None,
    save_path=None,
    title=None
):
    """
    Draw a horizontal bar chart from one heatmap column.

    heatmap_df: dataframe, returned by plot_v_region_heatmap_for_labeled_genes
                for example g1_tra_heatmap_labeled_df
    value_region: selects the overlap region; the first heatmap column is "100"
                  under the analysis definition this is "target group only", that is "only g1"
    """

    plot_df = heatmap_df.copy()

    if plot_df.empty:
        print("The heatmap table is empty; no bar plot was drawn")
        return pd.DataFrame()

    if "v" not in plot_df.columns:
        raise ValueError("The heatmap table lacks column 'v'. Pass the output of plot_v_region_heatmap_for_labeled_genes().")

    if value_region not in plot_df.columns:
        raise ValueError(
            f"The heatmap table lacks column {value_region}. "
            f"Available columns: {list(plot_df.columns)}"
        )

    plot_df[value_region] = pd.to_numeric(plot_df[value_region], errors="coerce")
    plot_df = plot_df.dropna(subset=[value_region]).copy()

    plot_df = plot_df.sort_values(value_region, ascending=False).reset_index(drop=True)

    if top_n is not None:
        plot_df = plot_df.head(top_n).copy()

    region_label = REGION_LABELS.get(value_region, value_region).replace(
        "target group",
        target_label
    )

    plt.rcParams["font.family"] = "DejaVu Sans"

    y = np.arange(len(plot_df))

    fig_height = max(5, 0.48 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(12, fig_height))

    bar_color = "#4C78A8"

    ax.barh(
        y,
        plot_df[value_region],
        height=0.58,
        color=bar_color,
        alpha=0.92,
        label=region_label
    )

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["v"], fontsize=10)

    ax.invert_yaxis()

    ax.set_xlabel("p_i: share of this V segment's clonotypes in the region", fontsize=12)
    ax.set_ylabel("TRA V segment", fontsize=12)

    if title is None:
        title = (
            f"TRA, {target_label}: V-segment contribution to region '{region_label}'"
        )

    ax.set_title(title, fontsize=14, pad=14)

    ax.xaxis.grid(True, linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    max_x = max(float(plot_df[value_region].max()), 0.01)

    for i, value in enumerate(plot_df[value_region]):
        ax.text(
            value + max_x * 0.015,
            i,
            f"{value:.2f}",
            va="center",
            ha="left",
            fontsize=9,
            color="#333333"
        )

    ax.set_xlim(0, max_x * 1.18)

    ax.legend(
        loc="lower right",
        frameon=False,
        fontsize=10
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return plot_df

g1_only_barplot_df = plot_heatmap_first_column_as_barh(
    heatmap_df=g1_tra_heatmap_labeled_df,
    target_label="g1",
    value_region="100",
    top_n=None,
    save_path=str(FIGURE_DIR / "TRA_g1_only_segment_barplot.png")
)

## TRB exact-set analysis


In [ ]:
ACTIVE_CHAIN = "TRB"
ct_g1_aaV_trb = intersections.count_table(
    clonosets_trb_g1,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g2_aaV_trb = intersections.count_table(
    clonosets_trb_g2,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g3_aaV_trb = intersections.count_table(
    clonosets_trb_g3,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g4_aaV_trb = intersections.count_table(
    clonosets_trb_g4,
    
    overlap_type="aaV",
    mismatches=0
)

ct_g5_aaV_trb = intersections.count_table(
    clonosets_trb_g5,
    
    overlap_type="aaV",
    mismatches=0
)













ct_g6_aaV_trb = intersections.count_table(
    clonosets_trb_g6,
    
    overlap_type="aaV",
    mismatches=0
)

### TRB overlap regions and retained V segments


In [ ]:
def plot_double_triple_venn_like(
    center_set,
    left_top_set,
    left_bottom_set,
    right_top_set,
    right_bottom_set,
    center_label,
    left_top_label,
    left_bottom_label,
    right_top_label,
    right_bottom_label,
    title=None,
    save_path=None
):
    """
    5-circle venn-like plot:
    
    The left triplet:  left_top, center, left_bottom
    The right triplet: right_top, center, right_bottom
    
    Cross-intersections between the left and right triplets
    (for example g5∩g2, g6∩g4 and related combinations) are not displayed.
    """

    center_set = set(center_set)
    left_top_set = set(left_top_set)
    left_bottom_set = set(left_bottom_set)
    right_top_set = set(right_top_set)
    right_bottom_set = set(right_bottom_set)

    
    
    
    left_top_only = left_top_set - (center_set | left_bottom_set)
    left_bottom_only = left_bottom_set - (center_set | left_top_set)
    left_top_left_bottom_only = (left_top_set & left_bottom_set) - center_set
    center_left_top_only = (center_set & left_top_set) - left_bottom_set
    center_left_bottom_only = (center_set & left_bottom_set) - left_top_set
    left_triple = center_set & left_top_set & left_bottom_set

    
    
    
    right_top_only = right_top_set - (center_set | right_bottom_set)
    right_bottom_only = right_bottom_set - (center_set | right_top_set)
    right_top_right_bottom_only = (right_top_set & right_bottom_set) - center_set
    center_right_top_only = (center_set & right_top_set) - right_bottom_set
    center_right_bottom_only = (center_set & right_bottom_set) - right_top_set
    right_triple = center_set & right_top_set & right_bottom_set

    
    
    
    center_only_after_all = center_set - (
        left_top_set | left_bottom_set | right_top_set | right_bottom_set
    )

    
    
    
    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.set_aspect("equal")
    ax.axis("off")

    
    colors = {
        left_top_label:   "#e41a1c",  
        center_label:     "#4daf4a",  
        left_bottom_label:"#377eb8",  
        right_top_label:  "#ff7f00",  
        right_bottom_label:"#984ea3", 
    }

    circles = {
        left_top_label: {
            "xy": (-2.0,  1.0),
            "r": 1.35,
            "set": left_top_set
        },
        left_bottom_label: {
            "xy": (-2.0, -1.0),
            "r": 1.35,
            "set": left_bottom_set
        },
        center_label: {
            "xy": (0.0, 0.0),
            "r": 1.55,
            "set": center_set
        },
        right_top_label: {
            "xy": (2.0,  1.0),
            "r": 1.35,
            "set": right_top_set
        },
        right_bottom_label: {
            "xy": (2.0, -1.0),
            "r": 1.35,
            "set": right_bottom_set
        }
    }

    
    for label, params in circles.items():
        c = Circle(
            params["xy"],
            params["r"],
            facecolor=colors[label],
            edgecolor="black",
            linewidth=2,
            alpha=0.28,
            zorder=1
        )
        ax.add_patch(c)

    
    
    
    
    label_positions = {
        left_top_label:    (-3.0,  1.65),
        left_bottom_label: (-3.0, -1.65),
        center_label:      ( 0.0, -1.55),
        right_top_label:   ( 3.0,  1.65),
        right_bottom_label:( 3.0, -1.65),
    }

    for label, (x, y) in label_positions.items():
        n = len(circles[label]["set"])
        ax.text(
            x, y,
            f"{label}\n{n}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            zorder=5
        )

    
    
    
    def put_count(x, y, value):
        ax.text(
            x, y,
            f"{len(value)}",
            ha="center",
            va="center",
            fontsize=12,
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.18",
                facecolor="white",
                edgecolor="none",
                alpha=0.75
            ),
            zorder=10
        )

    
    
    
    

    
    put_count(-2.55,  1.05, left_top_only)               
    put_count(-2.55, -1.05, left_bottom_only)            
    put_count(-2.05,  0.00, left_top_left_bottom_only)   
    put_count(-1.05,  0.62, center_left_top_only)        
    put_count(-1.05, -0.62, center_left_bottom_only)     

    
    put_count(-1.30,  0.00, left_triple)                 
    put_count( 0.00,  0.00, center_only_after_all)       
    put_count( 1.30,  0.00, right_triple)                

    
    put_count( 2.55,  1.05, right_top_only)              
    put_count( 2.55, -1.05, right_bottom_only)           
    put_count( 2.05,  0.00, right_top_right_bottom_only) 
    put_count( 1.05,  0.62, center_right_top_only)       
    put_count( 1.05, -0.62, center_right_bottom_only)    

    if title is not None:
        ax.set_title(title, fontsize=15, pad=20)

    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-2.5, 2.5)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return {
        "center_only_after_all": center_only_after_all,
        "left_triple": left_triple,
        "right_triple": right_triple,
        "left_regions": {
            "left_top_only": left_top_only,
            "left_bottom_only": left_bottom_only,
            "left_top_left_bottom_only": left_top_left_bottom_only,
            "center_left_top_only": center_left_top_only,
            "center_left_bottom_only": center_left_bottom_only,
            "left_triple": left_triple,
        },
        "right_regions": {
            "right_top_only": right_top_only,
            "right_bottom_only": right_bottom_only,
            "right_top_right_bottom_only": right_top_right_bottom_only,
            "center_right_top_only": center_right_top_only,
            "center_right_bottom_only": center_right_bottom_only,
            "right_triple": right_triple,
        }
    }

g1_trb_aaV = set(ct_g1_aaV_trb.index)
g2_trb_aaV = set(ct_g2_aaV_trb.index)
g3_trb_aaV = set(ct_g3_aaV_trb.index)
g4_trb_aaV = set(ct_g4_aaV_trb.index)
g5_trb_aaV = set(ct_g5_aaV_trb.index)
g6_trb_aaV = set(ct_g6_aaV_trb.index)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

res_g1 = plot_double_triple_venn_like(
    center_set=g1_trb_aaV,
    left_top_set=g5_trb_aaV,
    left_bottom_set=g6_trb_aaV,
    right_top_set=g2_trb_aaV,
    right_bottom_set=g4_trb_aaV,
    center_label="g1",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g2",
    right_bottom_label="g4",
    title="TRB clonotypes overlap: g5-g1-g6 and g2-g1-g4, aaV",
    save_path=str(FIGURE_DIR / "TRB_aaV_double_triple_g1.png")
)

res_g2 = plot_double_triple_venn_like(
    center_set=g2_trb_aaV,
    left_top_set=g5_trb_aaV,
    left_bottom_set=g6_trb_aaV,
    right_top_set=g1_trb_aaV,
    right_bottom_set=g3_trb_aaV,
    center_label="g2",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g1",
    right_bottom_label="g3",
    title="TRB clonotypes overlap: g5-g2-g6 and g1-g2-g3, aaV",
    save_path=str(FIGURE_DIR / "TRB_aaV_double_triple_g2.png")
)

In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    clone_set = sorted(clone_set)

    if overlap_type == "aaV":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def plot_v_distribution(df, title, top_n=20):
    if df.empty:
        print(f"{title}: empty dataframe")
        return

    vc = df["v"].value_counts()

    if top_n is not None:
        vc = vc.head(top_n)

    plt.figure(figsize=(12, 6))
    vc.plot(kind="bar")
    plt.title(title, fontsize=14)
    plt.xlabel("V gene", fontsize=12)
    plt.ylabel("Number of unique clonotypes", fontsize=12)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
    

g1 = set(ct_g1_aaV_trb.index)
g2 = set(ct_g2_aaV_trb.index)
g3 = set(ct_g3_aaV_trb.index)
g4 = set(ct_g4_aaV_trb.index)
g5 = set(ct_g5_aaV_trb.index)
g6 = set(ct_g6_aaV_trb.index)




g1_remaining_aaV_trb = g1 - (g2 | g4 | g5 | g6)




g2_remaining_aaV_trb = g2 - (g1 | g3 | g5 | g6)



clones_g1_remaining_aaV_trb_df = clone_set_to_df(
    g1_remaining_aaV_trb,
    overlap_type="aaV"
)

clones_g2_remaining_aaV_trb_df = clone_set_to_df(
    g2_remaining_aaV_trb,
    overlap_type="aaV"
)

plot_v_distribution(
    clones_g1_remaining_aaV_trb_df,
    "TRB: V genes among clonotypes remaining in g1 after subtracting g2, g4, g5, g6, aaV"
)

plot_v_distribution(
    clones_g2_remaining_aaV_trb_df,
    "TRB: V genes among clonotypes remaining in g2 after subtracting g1, g3, g5, g6, aaV"
)

### TRB V-segment retention tables


In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    if overlap_type == "aaV":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def make_v_summary_from_clone_set(clone_set, total_n, overlap_type="aaV"):
    df = clone_set_to_df(clone_set, overlap_type=overlap_type)

    if df.empty:
        return pd.DataFrame(columns=["v", "n_cdr3", "frequency", "frequency_pct"])

    out = (
        df["v"]
        .value_counts()
        .rename_axis("v")
        .reset_index(name="n_cdr3")
    )

    out["frequency"] = out["n_cdr3"] / total_n
    out["frequency_pct"] = 100 * out["frequency"]

    return out


def compare_full_vs_remaining_after_subtractions(
    target_ct,
    subtract_cts,
    target_label,
    overlap_type="aaV"
):
    target_set = set(target_ct.index)
    subtract_union = set().union(*[set(ct.index) for ct in subtract_cts])

    remaining_set = target_set - subtract_union

    full_v = make_v_summary_from_clone_set(
        target_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    remaining_v = make_v_summary_from_clone_set(
        remaining_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    full_v = full_v.rename(columns={
        "n_cdr3": f"cdr3_in_full_{target_label}",
        "frequency": f"frequency_in_full_{target_label}",
        "frequency_pct": f"frequency_in_full_{target_label}_pct"
    })

    remaining_v = remaining_v.rename(columns={
        "n_cdr3": f"cdr3_remaining_in_{target_label}",
        "frequency": f"frequency_remaining_in_{target_label}_relative_to_original",
        "frequency_pct": f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    })

    cmp_df = full_v.merge(remaining_v, on="v", how="left")

    remaining_count_col = f"cdr3_remaining_in_{target_label}"
    full_count_col = f"cdr3_in_full_{target_label}"

    remaining_freq_col = f"frequency_remaining_in_{target_label}_relative_to_original"
    remaining_freq_pct_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_freq_pct_col = f"frequency_in_full_{target_label}_pct"

    cmp_df[remaining_count_col] = cmp_df[remaining_count_col].fillna(0).astype(int)
    cmp_df[remaining_freq_col] = cmp_df[remaining_freq_col].fillna(0.0)
    cmp_df[remaining_freq_pct_col] = cmp_df[remaining_freq_pct_col].fillna(0.0)

    cmp_df["lost_cdr3"] = cmp_df[full_count_col] - cmp_df[remaining_count_col]

    cmp_df["frequency_change_pct"] = (
        cmp_df[full_freq_pct_col] - cmp_df[remaining_freq_pct_col]
    )

    cmp_df["retained_cdr3_share"] = (
        cmp_df[remaining_count_col] / cmp_df[full_count_col]
    )

    cmp_df["retained_cdr3_share_pct"] = (
        100 * cmp_df["retained_cdr3_share"]
    )

    cmp_df["loss_share"] = cmp_df["lost_cdr3"] / cmp_df[full_count_col]
    cmp_df["loss_share_pct"] = 100 * cmp_df["loss_share"]

    cmp_df_changed = (
        cmp_df[cmp_df["lost_cdr3"] != 0]
        .sort_values(
            ["loss_share", "lost_cdr3", full_count_col],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )

    return cmp_df, cmp_df_changed, remaining_set

cmp_g1_all, cmp_g1_changed_ranked, g1_remaining_aaV_trb = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g1_aaV_trb,
        subtract_cts=[
            ct_g2_aaV_trb,
            ct_g4_aaV_trb,
            ct_g5_aaV_trb,
            ct_g6_aaV_trb
        ],
        target_label="g1",
        overlap_type="aaV"
    )
)

display(cmp_g1_changed_ranked.head(20))

cmp_g1_changed_ranked.to_csv(
    RESULT_DIR / "TRB_cmp_g1_remaining_after_g2_g4_g5_g6_ranked.csv",
    index=False
)

cmp_g2_all, cmp_g2_changed_ranked, g2_remaining_aaV_trb = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g2_aaV_trb,
        subtract_cts=[
            ct_g1_aaV_trb,
            ct_g3_aaV_trb,
            ct_g5_aaV_trb,
            ct_g6_aaV_trb
        ],
        target_label="g2",
        overlap_type="aaV"
    )
)

display(cmp_g2_changed_ranked.head(20))

cmp_g2_changed_ranked.to_csv(
    RESULT_DIR / "TRB_cmp_g2_remaining_after_g1_g3_g5_g6_ranked.csv",
    index=False
)

The following table supports the V-segment retention figures.


In [ ]:
def make_clone_intersection_table_venn3(
    ct_a,
    ct_b,
    ct_c,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
):
    set_a = set(ct_a.index)
    set_b = set(ct_b.index)
    set_c = set(ct_c.index)

    all_clones = set_a | set_b | set_c

    sum_a = ct_a.sum(axis=1).astype(float).to_dict()
    sum_b = ct_b.sum(axis=1).astype(float).to_dict()
    sum_c = ct_c.sum(axis=1).astype(float).to_dict()

    total_a = float(ct_a.sum(axis=1).sum())
    total_b = float(ct_b.sum(axis=1).sum())
    total_c = float(ct_c.sum(axis=1).sum())

    region_names = {
        "100": f"{label_a} only",
        "010": f"{label_b} only",
        "001": f"{label_c} only",
        "110": f"({label_a}∩{label_b})-{label_c}",
        "101": f"({label_a}∩{label_c})-{label_b}",
        "011": f"({label_b}∩{label_c})-{label_a}",
        "111": f"{label_a}∩{label_b}∩{label_c}",
    }

    rows = []

    for clone in sorted(all_clones):
        in_a = clone in set_a
        in_b = clone in set_b
        in_c = clone in set_c

        region_id = (
            ("1" if in_a else "0") +
            ("1" if in_b else "0") +
            ("1" if in_c else "0")
        )

        if overlap_type == "aaV":
            cdr3aa, v = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "clone_id": clone,
            }

        elif overlap_type == "aaVJ":
            cdr3aa, v, j = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "j": j,
                "clone_id": clone,
            }

        else:
            raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")

        row_sum_a = float(sum_a.get(clone, 0.0))
        row_sum_b = float(sum_b.get(clone, 0.0))
        row_sum_c = float(sum_c.get(clone, 0.0))

        row.update({
            f"in_{label_a}": in_a,
            f"in_{label_b}": in_b,
            f"in_{label_c}": in_c,

            "region_id": region_id,
            "region_name": region_names[region_id],

            f"row_sum_{label_a}": row_sum_a,
            f"row_sum_{label_b}": row_sum_b,
            f"row_sum_{label_c}": row_sum_c,

            f"freq_vs_{label_a}": row_sum_a / total_a if total_a > 0 else np.nan,
            f"freq_vs_{label_b}": row_sum_b / total_b if total_b > 0 else np.nan,
            f"freq_vs_{label_c}": row_sum_c / total_c if total_c > 0 else np.nan,
        })

        row[f"freq_vs_{label_a}_pct"] = 100 * row[f"freq_vs_{label_a}"]
        row[f"freq_vs_{label_b}_pct"] = 100 * row[f"freq_vs_{label_b}"]
        row[f"freq_vs_{label_c}_pct"] = 100 * row[f"freq_vs_{label_c}"]

        rows.append(row)

    out = (
        pd.DataFrame(rows)
        .sort_values(
            ["region_id", f"row_sum_{label_a}", f"row_sum_{label_b}", f"row_sum_{label_c}", "v", "cdr3aa"],
            ascending=[True, False, False, False, True, True]
        )
        .reset_index(drop=True)
    )

    return out

g1_g5_g6_intersection_table = make_clone_intersection_table_venn3(
    ct_a=ct_g1_aaV_trb,
    ct_b=ct_g5_aaV_trb,
    ct_c=ct_g6_aaV_trb,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
)

display(g1_g5_g6_intersection_table.head(20))

### TRB V-segment retention figures


In [ ]:
def make_v_region_summary(intersection_df, freq_col):
    out = (
        intersection_df
        .groupby(["v", "region_id", "region_name"], as_index=False)
        .agg(
            n_clones=("clone_id", "nunique"),
            burden=(freq_col, "sum"),
            median_clone_freq=(freq_col, "median"),
            mean_clone_freq=(freq_col, "mean")
        )
    )

    out["region_share_within_v"] = (
        out["burden"] /
        out.groupby("v")["burden"].transform("sum")
    )

    return out

def make_v_profile(cmp_df, intersection_df, target_label, freq_col):
    region_summary = make_v_region_summary(intersection_df, freq_col=freq_col)

    share_wide = (
        region_summary
        .pivot(index="v", columns="region_id", values="region_share_within_v")
        .fillna(0)
    )

    
    p = share_wide.replace(0, np.nan)
    entropy = -(p * np.log2(p)).sum(axis=1).fillna(0) / np.log2(7)
    share_wide["region_entropy"] = entropy

    profile = cmp_df.merge(
        share_wide.reset_index(),
        on="v",
        how="left"
    ).fillna(0)

    
    profile["private_share"] = profile.get("100", 0)
    profile["triple_share"] = profile.get("111", 0)

    full_freq_col = f"frequency_in_full_{target_label}_pct"
    full_count_col = f"cdr3_in_full_{target_label}"

    profile["impact_score"] = (
        profile[full_freq_col] *
        profile["loss_share"]
    )

    profile["specific_score"] = (
        profile[full_freq_col] *
        profile["retained_cdr3_share"]
    )

    profile = profile.sort_values("impact_score", ascending=False).reset_index(drop=True)

    return profile, region_summary

g1_profile, g1_v_region_summary = make_v_profile(
    cmp_df=cmp_g1_all,                      
    intersection_df=g1_g5_g6_intersection_table,
    target_label="g1",
    freq_col="freq_vs_g1"
)

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"

REGION_LABELS = {
    "100": "target group only",
    "110": "target group ∩ g5",
    "101": "target group ∩ g6",
    "111": "target group ∩ g5 ∩ g6",
    "010": "g5 only",
    "011": "g5 ∩ g6",
    "001": "g6 only",
}

METRIC_LABELS = {
    "private_share": "share of target-group-only clonotypes",
    "triple_share": "share of clonotypes in the triple intersection",
    "region_entropy": "normalized entropy across overlap regions",
    "impact_score": "loss index",
    "specific_score": "specificity index",
}


def plot_v_dumbbell(profile, target_label, top_n=20, min_full_freq_pct=0.5):
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    remaining_freq_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_count_col = f"cdr3_in_full_{target_label}"
    remaining_count_col = f"cdr3_remaining_in_{target_label}"

    plot_df = profile.copy()

    
    plot_df = plot_df[plot_df[full_freq_col] >= min_full_freq_pct].copy()

    if plot_df.empty:
        print(
            f"No V segments with an initial share >= {min_full_freq_pct}% "
            f"for group {target_label}"
        )
        return pd.DataFrame()

    
    plot_df["count_loss"] = (
        plot_df[full_count_col] - plot_df[remaining_count_col]
    )

    
    plot_df["scaled_retention_score"] = (
        plot_df[full_count_col] / (1 + plot_df["count_loss"])
    )

    
    plot_df = (
        plot_df
        .sort_values(
            ["scaled_retention_score", full_count_col, remaining_count_col],
            ascending=[False, False, False]
        )
        .head(top_n)
        .sort_values("scaled_retention_score", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(plot_df))

    plt.figure(figsize=(10, max(8, 0.35 * len(plot_df))))

    for i, row in plot_df.iterrows():
        plt.plot(
            [row[remaining_freq_col], row[full_freq_col]],
            [i, i],
            linewidth=2,
            alpha=0.75
        )

    plt.scatter(
        plot_df[remaining_freq_col],
        y,
        s=65,
        label="retained after all subtractions"
    )

    plt.scatter(
        plot_df[full_freq_col],
        y,
        s=65,
        label="complete group"
    )

    plt.yticks(y, plot_df["v"])
    plt.xlabel("Clonotypes assigned to the V segment, %")
    plt.ylabel("V segment")
    plt.title(
        f"TRB, {target_label}: V segments with an initial share ≥ {min_full_freq_pct}% "
        f"and minimal loss"
    )
    plt.legend(title="Set state")
    plt.tight_layout()
    plt.show()

    return plot_df
    
def get_bubble_labeled_v_genes(profile, target_label, top_n=10):
    full_count_col = f"cdr3_in_full_{target_label}"
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    required = {
        "v", "impact_score", "retained_cdr3_share_pct",
        full_count_col, full_freq_col,
    }
    missing = sorted(required.difference(profile.columns))
    if missing:
        raise ValueError(f"The V-segment profile is missing columns: {missing}")

    candidates = profile.copy()
    numeric_columns = [
        "impact_score", "retained_cdr3_share_pct", full_count_col, full_freq_col
    ]
    for column in numeric_columns:
        candidates[column] = pd.to_numeric(candidates[column], errors="coerce")
    candidates = candidates[
        candidates[full_count_col].gt(0)
        & candidates[full_freq_col].gt(0)
        & candidates["retained_cdr3_share_pct"].notna()
        & candidates["impact_score"].notna()
    ].copy()
    if candidates.empty:
        raise ValueError(
            f"No positive target-group V segments were available for {target_label}."
        )

    label_df = (
        candidates.sort_values(
            ["impact_score", "retained_cdr3_share_pct", full_count_col, "v"],
            ascending=[False, False, False, True],
        )
        .head(int(top_n))
        .reset_index(drop=True)
    )
    return label_df["v"].tolist(), label_df

    
def plot_v_landscape(
    profile,
    target_label,
    color_by="region_entropy",
    retained_threshold=None,
    top_n_labels=10,
):
    chain_label = "TRB"
    full_freq_col = f"frequency_in_full_{target_label}_pct"
    full_count_col = f"cdr3_in_full_{target_label}"

    plot_df = profile.copy()
    for column in [full_freq_col, full_count_col, "retained_cdr3_share_pct", color_by]:
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")
    plot_df = plot_df[
        plot_df[full_count_col].gt(0)
        & plot_df[full_freq_col].gt(0)
        & plot_df["retained_cdr3_share_pct"].notna()
        & plot_df[color_by].notna()
    ].copy()
    if plot_df.empty:
        raise ValueError(
            f"No positive target-group V segments were available for {target_label}."
        )

    labeled_genes, label_df = get_bubble_labeled_v_genes(
        profile=plot_df,
        target_label=target_label,
        top_n=top_n_labels,
    )
    maximum_count = float(plot_df[full_count_col].max())
    marker_size = 40 + 500 * plot_df[full_count_col] / maximum_count

    fig, ax = plt.subplots(figsize=(11, 8))
    scatter = ax.scatter(
        plot_df[full_freq_col],
        plot_df["retained_cdr3_share_pct"],
        c=plot_df[color_by],
        s=marker_size,
        alpha=0.78,
    )

    y_min = float(plot_df["retained_cdr3_share_pct"].min())
    y_max = float(plot_df["retained_cdr3_share_pct"].max())
    y_span = max(y_max - y_min, 1.0)
    y_padding = max(0.08 * y_span, 2.0)
    ax.set_ylim(max(0.0, y_min - y_padding), min(105.0, y_max + y_padding))

    x_min = float(plot_df[full_freq_col].min())
    x_max = float(plot_df[full_freq_col].max())
    x_span = max(x_max - x_min, 0.1)
    x_padding = max(0.08 * x_span, 0.03)
    ax.set_xlim(max(0.0, x_min - 0.2 * x_padding), x_max + x_padding)

    if retained_threshold is not None and y_min <= retained_threshold <= y_max:
        ax.axhline(
            retained_threshold,
            linestyle=":",
            linewidth=1.3,
            color="#4C78A8",
            alpha=0.8,
        )

    texts = []
    for row in label_df.itertuples():
        texts.append(
            ax.text(
                getattr(row, full_freq_col),
                row.retained_cdr3_share_pct,
                row.v,
                fontsize=9,
                ha="left",
                va="bottom",
                bbox=dict(
                    boxstyle="round,pad=0.18",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.78,
                ),
            )
        )
    if texts:
        adjust_text(
            texts,
            x=label_df[full_freq_col].values,
            y=label_df["retained_cdr3_share_pct"].values,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.6, alpha=0.7),
            expand=(1.2, 1.4),
            force_static=0.8,
            force_text=1.0,
            iter_lim=700,
            ax=ax,
        )

    ax.set_xlabel(f"V-segment share in the complete group {target_label}, %")
    ax.set_ylabel("Retained clonotypes, %")
    ax.set_title(
        f"{chain_label} | treatment group {target_label} | {STRATUM_LABEL}\n"
        f"V-segment retention landscape; top {len(label_df)} labels by impact score"
    )
    ax.grid(False)
    fig.colorbar(
        scatter,
        ax=ax,
        pad=0.03,
        label=METRIC_LABELS.get(color_by, color_by),
    )
    fig.tight_layout()
    plt.show()
    return labeled_genes, label_df

    

def plot_v_region_heatmap_for_labeled_genes(
    profile,
    labeled_genes,
    target_label,
    save_path=None,
):
    chain_label = "TRB"
    region_order = ["100", "110", "101", "111", "010", "011", "001"]
    plot_df = profile.copy()
    for region in region_order:
        if region not in plot_df.columns:
            plot_df[region] = 0.0

    selected_genes = list(dict.fromkeys(labeled_genes))[:10]
    order_map = {gene: rank for rank, gene in enumerate(selected_genes)}
    plot_df = plot_df[plot_df["v"].isin(selected_genes)].copy()
    if plot_df.empty:
        raise ValueError(
            f"No bubble-selected V segments were available for the {chain_label} heatmap."
        )
    plot_df["bubble_order"] = plot_df["v"].map(order_map)
    plot_df = plot_df.sort_values("bubble_order").set_index("v")
    heatmap = plot_df[region_order].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    x_labels = [
        REGION_LABELS[region].replace("target group", target_label)
        for region in region_order
    ]

    fig_height = max(5.0, 0.48 * len(heatmap) + 2.0)
    fig, ax = plt.subplots(figsize=(14, fig_height))
    image = ax.imshow(
        heatmap.values,
        aspect="auto",
        interpolation="none",
        cmap="magma",
        vmin=0.0,
        vmax=max(1.0, float(np.nanmax(heatmap.values))),
    )
    ax.set_xticks(range(len(region_order)), labels=x_labels, rotation=35, ha="right")
    ax.set_yticks(range(len(heatmap.index)), labels=heatmap.index)
    ax.grid(False)
    ax.tick_params(which="minor", bottom=False, left=False)

    for row_index in range(heatmap.shape[0]):
        for column_index in range(heatmap.shape[1]):
            value = float(heatmap.iloc[row_index, column_index])
            ax.text(
                column_index,
                row_index,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=_adaptive_annotation_color(image, value),
            )

    ax.set_xlabel("Overlap region")
    ax.set_ylabel(f"{chain_label} V segment")
    ax.set_title(
        f"{chain_label} | treatment group {target_label} | {STRATUM_LABEL}\n"
        f"Overlap architecture of the top {len(heatmap)} bubble-plot V segments"
    )
    fig.colorbar(
        image,
        ax=ax,
        pad=0.03,
        label="Share of each V segment's clonotypes in the overlap region",
    )
    fig.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return plot_df.reset_index()



g1_trb_dumbbell_selected_df = plot_v_dumbbell(
    g1_profile,
    target_label="g1",
    top_n=20,
    min_full_freq_pct=1
)

g1_labeled_genes, g1_bubble_labels_df = plot_v_landscape(
    g1_profile,
    target_label="g1",
    color_by="region_entropy",
    retained_threshold=59,
    top_n_labels=10,
)


g1_heatmap_labeled_df = plot_v_region_heatmap_for_labeled_genes(
    g1_profile,
    labeled_genes=g1_labeled_genes,
    target_label="g1",
    save_path=str(FIGURE_DIR / "TRB_g1_top10_bubble_genes_heatmap.png"),
)





## TRA/TRB comparison using Jensen-Shannon similarity and rank correlation


In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
from itertools import combinations
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon
from adjustText import adjust_text

plt.rcParams["font.family"] = "DejaVu Sans"

out_dir = str(RESULT_DIR / "tra_trb_pairwise_comparison")
os.makedirs(out_dir, exist_ok=True)

ct_aaV_by_group_chain = {
    "g1": {
        "TRA": ct_g1_aaV_tra,
        "TRB": ct_g1_aaV_trb,
    },
    "g2": {
        "TRA": ct_g2_aaV_tra,
        "TRB": ct_g2_aaV_trb,
    },
    "g5": {
        "TRA": ct_g5_aaV_tra,
        "TRB": ct_g5_aaV_trb,
    },
}

def infer_mouse_id_from_sample_id(sample_id):
    
    sample_id = str(sample_id)
    m = re.search(r"(g\d+)_m(\d+)", sample_id)

    if m is None:
        return sample_id

    return f"{m.group(1)}_m{m.group(2)}"


def prepare_metadata_mouse_id(metadata):
    md = metadata.copy()

    if "mouse_id" not in md.columns:
        md["mouse_id"] = md["sample_id"].apply(infer_mouse_id_from_sample_id)

    md["sample_id"] = md["sample_id"].astype(str)
    md["mouse_id"] = md["mouse_id"].astype(str)

    return md


def index_value_to_tuple(x):
    
    if isinstance(x, tuple):
        return x

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, tuple):
                return y
        except Exception:
            pass

    return x


def get_v_from_index(ct):
    
    if isinstance(ct.index, pd.MultiIndex):
        return ct.index.get_level_values(1).astype(str).tolist()

    out = []
    for x in ct.index:
        x = index_value_to_tuple(x)

        if not isinstance(x, tuple) or len(x) < 2:
            raise ValueError(
                "Cannot extract the V segment. "
                "Expected an aaV index of (cdr3aa, v)"
            )

        out.append(str(x[1]))

    return out


def make_feature_by_sample_matrix(ct, feature_mode="v"):
    
    x = ct.copy()
    x = x.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    if feature_mode == "aaV":
        x.index = [index_value_to_tuple(i) for i in x.index]
        return x

    if feature_mode == "v":
        v_genes = get_v_from_index(x)
        x["feature_id"] = v_genes

        out = (
            x
            .groupby("feature_id")
            .sum()
        )

        return out

    raise ValueError("feature_mode must be 'v' or 'aaV'")
    
def aggregate_samples_to_mouse(feature_by_sample, metadata):
    
    md = prepare_metadata_mouse_id(metadata)

    sample_to_mouse = dict(zip(md["sample_id"], md["mouse_id"]))

    mat = feature_by_sample.copy()
    mat.columns = mat.columns.astype(str)

    
    for sample_id in mat.columns:
        if sample_id not in sample_to_mouse:
            sample_to_mouse[sample_id] = infer_mouse_id_from_sample_id(sample_id)

    mouse_cols = [sample_to_mouse[c] for c in mat.columns]

    mat_by_mouse = mat.copy()
    mat_by_mouse.columns = mouse_cols

    
    mat_by_mouse = mat_by_mouse.T.groupby(level=0).sum()

    return mat_by_mouse


def normalize_rows_to_freq(mat):
    
    mat = mat.copy().astype(float)
    row_sums = mat.sum(axis=1)

    freq = mat.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)

    return freq

def pair_similarity(x, y, metric="js"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if metric == "js":
        
        
        
        
        if x.sum() == 0 or y.sum() == 0:
            return np.nan

        x = x / x.sum()
        y = y / y.sum()

        return 1.0 - jensenshannon(x, y, base=2)

    if metric == "pearson":
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return pearsonr(x, y)[0]

    if metric == "spearman":
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y)[0]

    if metric == "jaccard":
        xb = x > 0
        yb = y > 0

        union = np.logical_or(xb, yb).sum()
        inter = np.logical_and(xb, yb).sum()

        if union == 0:
            return np.nan

        return inter / union

    raise ValueError("metric must be 'js', 'pearson', 'spearman' or 'jaccard'")


def compute_similarity_matrix(mouse_feature_mat, metric="js"):
    ids = list(mouse_feature_mat.index)
    sim = pd.DataFrame(np.nan, index=ids, columns=ids)

    for a in ids:
        for b in ids:
            if a == b:
                sim.loc[a, b] = 1.0
            elif pd.isna(sim.loc[a, b]):
                value = pair_similarity(
                    mouse_feature_mat.loc[a].values,
                    mouse_feature_mat.loc[b].values,
                    metric=metric
                )
                sim.loc[a, b] = value
                sim.loc[b, a] = value

    return sim


def similarity_matrix_to_pairs(sim_mat, chain_label):
    rows = []

    for a, b in combinations(sim_mat.index, 2):
        rows.append({
            "mouse_a": a,
            "mouse_b": b,
            "pair_id": tuple(sorted((a, b))),
            f"similarity_{chain_label}": sim_mat.loc[a, b]
        })

    return pd.DataFrame(rows)

def compare_tra_trb_from_count_tables(
    ct_tra,
    ct_trb,
    metadata,
    group_no,
    feature_mode="v",
    metric="js"
):
    

    tra_feature_sample = make_feature_by_sample_matrix(
        ct_tra,
        feature_mode=feature_mode
    )

    trb_feature_sample = make_feature_by_sample_matrix(
        ct_trb,
        feature_mode=feature_mode
    )

    tra_mouse = aggregate_samples_to_mouse(
        tra_feature_sample,
        metadata=metadata
    )

    trb_mouse = aggregate_samples_to_mouse(
        trb_feature_sample,
        metadata=metadata
    )

    common_mice = sorted(set(tra_mouse.index) & set(trb_mouse.index))

    if len(common_mice) < 2:
        raise ValueError(
            f"{group_no}: fewer than two mice are shared between TRA and TRB. "
            f"TRA mice: {list(tra_mouse.index)}. "
            f"TRB mice: {list(trb_mouse.index)}."
        )

    tra_mouse = tra_mouse.loc[common_mice]
    trb_mouse = trb_mouse.loc[common_mice]

    if metric in ["js", "pearson", "spearman"]:
        tra_for_similarity = normalize_rows_to_freq(tra_mouse)
        trb_for_similarity = normalize_rows_to_freq(trb_mouse)
    else:
        tra_for_similarity = tra_mouse.copy()
        trb_for_similarity = trb_mouse.copy()

    sim_tra = compute_similarity_matrix(
        tra_for_similarity,
        metric=metric
    )

    sim_trb = compute_similarity_matrix(
        trb_for_similarity,
        metric=metric
    )

    pairs_tra = similarity_matrix_to_pairs(sim_tra, "TRA")
    pairs_trb = similarity_matrix_to_pairs(sim_trb, "TRB")

    pair_df = pairs_tra.merge(
        pairs_trb[["pair_id", "similarity_TRB"]],
        on="pair_id",
        how="inner"
    )

    ok = pair_df[["similarity_TRA", "similarity_TRB"]].dropna()

    if len(ok) >= 2:
        pearson_r, pearson_p = pearsonr(ok["similarity_TRA"], ok["similarity_TRB"])
        spearman_rho, spearman_p = spearmanr(ok["similarity_TRA"], ok["similarity_TRB"])
    else:
        pearson_r, pearson_p = np.nan, np.nan
        spearman_rho, spearman_p = np.nan, np.nan

    summary = pd.DataFrame([{
        "group_no": group_no,
        "feature_mode": feature_mode,
        "metric": metric,
        "n_common_mice": len(common_mice),
        "n_pairs": len(ok),
        "mean_similarity_TRA": ok["similarity_TRA"].mean(),
        "mean_similarity_TRB": ok["similarity_TRB"].mean(),
        "median_similarity_TRA": ok["similarity_TRA"].median(),
        "median_similarity_TRB": ok["similarity_TRB"].median(),
        "TRA_vs_TRB_pearson_r": pearson_r,
        "TRA_vs_TRB_pearson_p": pearson_p,
        "TRA_vs_TRB_spearman_rho": spearman_rho,
        "TRA_vs_TRB_spearman_p": spearman_p,
    }])

    return {
        "group_no": group_no,
        "feature_mode": feature_mode,
        "metric": metric,
        "common_mice": common_mice,
        "matrix_TRA": tra_for_similarity,
        "matrix_TRB": trb_for_similarity,
        "sim_TRA": sim_tra,
        "sim_TRB": sim_trb,
        "pair_df": pair_df,
        "summary": summary,
    }

def plot_tra_trb_similarity_heatmaps(result, save_path=None):
    sim_tra = result["sim_TRA"]
    sim_trb = result["sim_TRB"]
    group_no = result["group_no"]
    feature_mode = result["feature_mode"]
    metric = result["metric"]
    vmin, vmax = (-1, 1) if metric in ["pearson", "spearman"] else (0, 1)

    fig = plt.figure(figsize=(15, 6.5), constrained_layout=True)
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.045])
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
    color_axis = fig.add_subplot(grid[0, 2])

    images = []
    for axis, matrix, chain_label in zip(axes, [sim_tra, sim_trb], ["TRA", "TRB"]):
        image = axis.imshow(
            matrix.values,
            aspect="equal",
            interpolation="none",
            vmin=vmin,
            vmax=vmax,
            cmap="magma",
        )
        images.append(image)
        axis.set_title(f"{chain_label}: {feature_mode} features; {metric} similarity")
        axis.set_xticks(
            range(len(matrix.columns)), labels=matrix.columns, rotation=45, ha="right"
        )
        axis.set_yticks(range(len(matrix.index)), labels=matrix.index)
        axis.grid(False)
        axis.tick_params(which="minor", bottom=False, left=False)
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                value = float(matrix.iloc[row_index, column_index])
                if np.isfinite(value):
                    axis.text(
                        column_index,
                        row_index,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color=_adaptive_annotation_color(image, value),
                    )

    colorbar = fig.colorbar(images[-1], cax=color_axis)
    colorbar.set_label("Similarity between mice")
    fig.suptitle(
        f"TRA and TRB | treatment group {group_no} | {STRATUM_LABEL}\n"
        "Pairwise mouse similarity by V-segment usage",
        fontsize=14,
    )
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    
def plot_tra_trb_pair_scatter(result, save_path=None):
    pair_df = result["pair_df"].copy()
    summary = result["summary"].iloc[0]

    group_no = result["group_no"]
    feature_mode = result["feature_mode"]
    metric = result["metric"]

    plt.figure(figsize=(8, 8))

    plt.scatter(
        pair_df["similarity_TRA"],
        pair_df["similarity_TRB"],
        s=80,
        alpha=0.8
    )

    vals = pd.concat([
        pair_df["similarity_TRA"],
        pair_df["similarity_TRB"]
    ]).dropna()

    mn = vals.min()
    mx = vals.max()

    plt.plot(
        [mn, mx],
        [mn, mx],
        linestyle=":",
        linewidth=1.5,
        label="TRA = TRB"
    )

    texts = []

    for _, row in pair_df.iterrows():
        txt = plt.text(
            row["similarity_TRA"],
            row["similarity_TRB"],
            f"{row['mouse_a']}–{row['mouse_b']}",
            fontsize=8
        )
        texts.append(txt)

    if len(texts) > 0:
        adjust_text(
            texts,
            arrowprops=dict(
                arrowstyle="-",
                color="gray",
                lw=0.5,
                alpha=0.6
            ),
            iter_lim=500
        )

    plt.xlabel("Mouse-pair similarity for TRA")
    plt.ylabel("Mouse-pair similarity for TRB")
    plt.title(
        f"{group_no}: TRA/TRB concordance\n"
        f"mode: {feature_mode}, metric: {metric}\n"
        f"Spearman rho = {summary['TRA_vs_TRB_spearman_rho']:.3f}"
    )

    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    
def collect_pairwise_similarity_boxplot_table(results_dict):
    rows = []

    for group_no, result in results_dict.items():
        pair_df = result["pair_df"].copy()

        for value in pair_df["similarity_TRA"].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRA",
                "similarity": value,
                "feature_mode": result["feature_mode"],
                "metric": result["metric"]
            })

        for value in pair_df["similarity_TRB"].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRB",
                "similarity": value,
                "feature_mode": result["feature_mode"],
                "metric": result["metric"]
            })

    return pd.DataFrame(rows)


def plot_group_chain_boxplot(box_df, title=None, save_path=None):
    labels = []
    data = []

    for group_no in sorted(box_df["group_no"].unique()):
        for chain in ["TRA", "TRB"]:
            values = box_df[
                (box_df["group_no"] == group_no) &
                (box_df["chain"] == chain)
            ]["similarity"].dropna().values

            labels.append(f"{group_no}\n{chain}")
            data.append(values)

    plt.figure(figsize=(10, 6))
    plt.boxplot(data, labels=labels)

    plt.ylabel("Within-group pairwise mouse similarity")

    if title is None:
        title = "Within-group TRA and TRB similarity"

    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
groups = ["g1", "g2", "g5"]
results_v_js = {}
similarity_status = []

for group_no in groups:
    try:
        result = compare_tra_trb_from_count_tables(
            ct_tra=globals()[f"ct_{group_no}_aaV_tra"],
            ct_trb=globals()[f"ct_{group_no}_aaV_trb"],
            metadata=metadata,
            group_no=group_no,
            feature_mode="v",
            metric="js",
        )
    except ValueError as error:
        similarity_status.append(
            {"group_no": group_no, "status": "not estimated", "reason": str(error)}
        )
        print(f"Skipping {group_no} TRA/TRB similarity: {error}")
        continue

    results_v_js[group_no] = result
    similarity_status.append(
        {"group_no": group_no, "status": "estimated", "reason": ""}
    )
    display(result["summary"])
    result["summary"].to_csv(
        f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_summary.csv", index=False
    )
    result["pair_df"].to_csv(
        f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_pairwise.csv", index=False
    )
    plot_tra_trb_similarity_heatmaps(
        result,
        save_path=f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_heatmaps.png",
    )
    plot_tra_trb_pair_scatter(
        result,
        save_path=f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_scatter.png",
    )

pd.DataFrame(similarity_status).to_csv(
    f"{out_dir}/TRA_TRB_V_usage_JS_status.csv", index=False
)
if results_v_js:
    box_v_js = collect_pairwise_similarity_boxplot_table(results_v_js)
    box_v_js.to_csv(
        f"{out_dir}/all_groups_TRA_TRB_V_usage_JS_boxplot_table.csv", index=False
    )
    if not box_v_js.empty:
        plot_group_chain_boxplot(
            box_v_js,
            title="TRA and TRB within-group mouse similarity by V-segment usage",
            save_path=f"{out_dir}/all_groups_TRA_TRB_V_usage_JS_boxplot.png",
        )


## Mouse-level TRA/TRB comparison using UMI counts


In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
import os
import re
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from scipy.stats import pearsonr, spearmanr
from adjustText import adjust_text

plt.rcParams["font.family"] = "DejaVu Sans"






OUT_DIR = str(RESULT_DIR / "tra_trb_pairwise_umi_clone_correlations")
os.makedirs(OUT_DIR, exist_ok=True)

GROUPS_TO_RUN = ["g1", "g2", "g5", "g6"]






TRANSFORM = "log1p"


MIN_CLONES_UNION = 10

SHOW_FIGURES = True






def get_existing_var(var_name):
    return globals().get(var_name, None)


def build_ct_pair_plan(groups_to_run):
    
    plan = {}

    for group_no in groups_to_run:
        ct_tra_name = f"ct_{group_no}_aaV_tra"
        ct_trb_name = f"ct_{group_no}_aaV_trb"

        ct_tra = get_existing_var(ct_tra_name)
        ct_trb = get_existing_var(ct_trb_name)

        if ct_tra is None:
            print(f"Skipping {group_no}: missing {ct_tra_name}")
            continue

        if ct_trb is None:
            print(f"Skipping {group_no}: missing {ct_trb_name}")
            continue

        plan[group_no] = {
            "TRA": ct_tra,
            "TRB": ct_trb,
            "ct_tra_name": ct_tra_name,
            "ct_trb_name": ct_trb_name,
        }

    return plan






def infer_mouse_id_from_sample_id(sample_id):
    
    sample_id = str(sample_id)
    m = re.search(r"(g\d+)_m(\d+)", sample_id)

    if m is None:
        return sample_id

    return f"{m.group(1)}_m{m.group(2)}"


def prepare_metadata_mouse_id(metadata):
    md = metadata.copy()

    if "sample_id" not in md.columns:
        raise ValueError("Metadata does not contain a sample_id column")

    md["sample_id"] = md["sample_id"].astype(str)

    if "mouse_id" not in md.columns:
        md["mouse_id"] = md["sample_id"].apply(infer_mouse_id_from_sample_id)
    else:
        md["mouse_id"] = md["mouse_id"].astype(str)

    return md


def parse_index_value(x):
    
    if isinstance(x, tuple):
        return x

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, tuple):
                return y
        except Exception:
            pass

    return x


def get_clone_ids_from_count_table(ct):
    
    if isinstance(ct.index, pd.MultiIndex):
        clone_ids = list(ct.index.to_flat_index())
    else:
        clone_ids = [parse_index_value(x) for x in ct.index]

    return clone_ids


def count_table_to_mouse_clone_count_matrix(ct, metadata, group_no=None):
    
    md = prepare_metadata_mouse_id(metadata)

    if group_no is not None and "group_no" in md.columns:
        md = md[md["group_no"] == group_no].copy()

    sample_to_mouse = dict(zip(md["sample_id"], md["mouse_id"]))

    x = ct.copy()
    x.columns = x.columns.astype(str)

    sample_cols = [c for c in x.columns if c in sample_to_mouse]

    
    
    if len(sample_cols) == 0:
        sample_cols = list(x.columns)
        sample_to_mouse = {
            sample_id: infer_mouse_id_from_sample_id(sample_id)
            for sample_id in sample_cols
        }

    x = x[sample_cols].copy()
    x = x.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    clone_ids = get_clone_ids_from_count_table(x)
    x.index = pd.Index(clone_ids, dtype="object")

    x.columns = [sample_to_mouse[c] for c in x.columns]

    
    mouse_clone_mat = x.T.groupby(level=0).sum().T

    return mouse_clone_mat






def transform_pair_vectors(x, y, transform="log1p"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if transform == "raw":
        return x, y

    if transform == "log1p":
        return np.log1p(x), np.log1p(y)

    if transform == "freq":
        sx = x.sum()
        sy = y.sum()

        x = x / sx if sx > 0 else x
        y = y / sy if sy > 0 else y

        return x, y

    if transform == "log1p_freq":
        sx = x.sum()
        sy = y.sum()

        x = x / sx if sx > 0 else x
        y = y / sy if sy > 0 else y

        return np.log1p(x), np.log1p(y)

    raise ValueError("transform must be one of: raw, log1p, freq, log1p_freq")


def safe_corr(x, y, method="pearson"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    if method == "pearson":
        r, p = pearsonr(x, y)
        return r, p

    if method == "spearman":
        rho, p = spearmanr(x, y)
        return rho, p

    raise ValueError("method must be pearson or spearman")


def compute_pairwise_clone_umi_correlations(
    mouse_clone_mat,
    chain_label,
    group_no,
    transform="log1p",
    min_clones_union=10
):
    
    mouse_ids = sorted(mouse_clone_mat.columns.astype(str))
    mat = mouse_clone_mat.copy()
    mat.columns = mat.columns.astype(str)

    rows = []

    for mouse_a, mouse_b in combinations(mouse_ids, 2):
        raw_a = mat[mouse_a].astype(float).values
        raw_b = mat[mouse_b].astype(float).values

        
        
        union_mask = (raw_a > 0) | (raw_b > 0)
        shared_mask = (raw_a > 0) & (raw_b > 0)

        n_union = int(union_mask.sum())
        n_shared = int(shared_mask.sum())

        if n_union < min_clones_union:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_rho, spearman_p = np.nan, np.nan
        else:
            x = raw_a[union_mask]
            y = raw_b[union_mask]

            x_t, y_t = transform_pair_vectors(
                x,
                y,
                transform=transform
            )

            pearson_r, pearson_p = safe_corr(
                x_t,
                y_t,
                method="pearson"
            )

            spearman_rho, spearman_p = safe_corr(
                x_t,
                y_t,
                method="spearman"
            )

        rows.append({
            "group_no": group_no,
            "chain": chain_label,
            "mouse_a": mouse_a,
            "mouse_b": mouse_b,
            "pair_id": tuple(sorted((mouse_a, mouse_b))),
            "transform": transform,
            "n_clones_union": n_union,
            "n_clones_shared": n_shared,
            "pct_shared_among_union": 100 * n_shared / n_union if n_union > 0 else np.nan,
            "pearson_umi_corr": pearson_r,
            "pearson_umi_p": pearson_p,
            "spearman_umi_rho": spearman_rho,
            "spearman_umi_p": spearman_p,
        })

    return pd.DataFrame(rows)


def pair_df_to_similarity_matrix(pair_df, value_col):
    mice = sorted(set(pair_df["mouse_a"]) | set(pair_df["mouse_b"]))

    sim = pd.DataFrame(
        np.eye(len(mice)),
        index=mice,
        columns=mice,
        dtype=float
    )

    for _, row in pair_df.iterrows():
        a = row["mouse_a"]
        b = row["mouse_b"]
        value = row[value_col]

        sim.loc[a, b] = value
        sim.loc[b, a] = value

    return sim


def compare_tra_trb_umi_correlations_for_group(
    ct_tra,
    ct_trb,
    metadata,
    group_no,
    transform="log1p",
    min_clones_union=10
):
    tra_mat = count_table_to_mouse_clone_count_matrix(
        ct=ct_tra,
        metadata=metadata,
        group_no=group_no
    )

    trb_mat = count_table_to_mouse_clone_count_matrix(
        ct=ct_trb,
        metadata=metadata,
        group_no=group_no
    )

    common_mice = sorted(
        set(tra_mat.columns.astype(str)) &
        set(trb_mat.columns.astype(str))
    )

    if len(common_mice) < 2:
        raise ValueError(
            f"{group_no}: fewer than two mice are shared between TRA and TRB. "
            f"TRA mice: {list(tra_mat.columns)}. "
            f"TRB mice: {list(trb_mat.columns)}."
        )

    tra_mat = tra_mat[common_mice]
    trb_mat = trb_mat[common_mice]

    tra_pairs = compute_pairwise_clone_umi_correlations(
        mouse_clone_mat=tra_mat,
        chain_label="TRA",
        group_no=group_no,
        transform=transform,
        min_clones_union=min_clones_union
    )

    trb_pairs = compute_pairwise_clone_umi_correlations(
        mouse_clone_mat=trb_mat,
        chain_label="TRB",
        group_no=group_no,
        transform=transform,
        min_clones_union=min_clones_union
    )

    pair_df = tra_pairs.merge(
        trb_pairs[
            [
                "pair_id",
                "n_clones_union",
                "n_clones_shared",
                "pct_shared_among_union",
                "pearson_umi_corr",
                "pearson_umi_p",
                "spearman_umi_rho",
                "spearman_umi_p"
            ]
        ],
        on="pair_id",
        how="inner",
        suffixes=("_TRA", "_TRB")
    )

    
    
    

    ok_pearson = pair_df[
        ["pearson_umi_corr_TRA", "pearson_umi_corr_TRB"]
    ].dropna()

    if len(ok_pearson) >= 2:
        tra_trb_pearson_r_on_paircorr, tra_trb_pearson_p_on_paircorr = pearsonr(
            ok_pearson["pearson_umi_corr_TRA"],
            ok_pearson["pearson_umi_corr_TRB"]
        )

        tra_trb_spearman_rho_on_paircorr, tra_trb_spearman_p_on_paircorr = spearmanr(
            ok_pearson["pearson_umi_corr_TRA"],
            ok_pearson["pearson_umi_corr_TRB"]
        )
    else:
        tra_trb_pearson_r_on_paircorr = np.nan
        tra_trb_pearson_p_on_paircorr = np.nan
        tra_trb_spearman_rho_on_paircorr = np.nan
        tra_trb_spearman_p_on_paircorr = np.nan

    summary = pd.DataFrame([{
        "group_no": group_no,
        "transform": transform,
        "n_common_mice": len(common_mice),
        "n_pairs": len(pair_df),

        "mean_pearson_umi_corr_TRA": pair_df["pearson_umi_corr_TRA"].mean(),
        "mean_pearson_umi_corr_TRB": pair_df["pearson_umi_corr_TRB"].mean(),

        "median_pearson_umi_corr_TRA": pair_df["pearson_umi_corr_TRA"].median(),
        "median_pearson_umi_corr_TRB": pair_df["pearson_umi_corr_TRB"].median(),

        "mean_spearman_umi_rho_TRA": pair_df["spearman_umi_rho_TRA"].mean(),
        "mean_spearman_umi_rho_TRB": pair_df["spearman_umi_rho_TRB"].mean(),

        "median_spearman_umi_rho_TRA": pair_df["spearman_umi_rho_TRA"].median(),
        "median_spearman_umi_rho_TRB": pair_df["spearman_umi_rho_TRB"].median(),

        "TRA_vs_TRB_pearson_r_of_pairwise_pearson_corr": tra_trb_pearson_r_on_paircorr,
        "TRA_vs_TRB_pearson_p_of_pairwise_pearson_corr": tra_trb_pearson_p_on_paircorr,

        "TRA_vs_TRB_spearman_rho_of_pairwise_pearson_corr": tra_trb_spearman_rho_on_paircorr,
        "TRA_vs_TRB_spearman_p_of_pairwise_pearson_corr": tra_trb_spearman_p_on_paircorr,
    }])

    sim_tra_pearson = pair_df_to_similarity_matrix(
        pair_df,
        "pearson_umi_corr_TRA"
    )

    sim_trb_pearson = pair_df_to_similarity_matrix(
        pair_df,
        "pearson_umi_corr_TRB"
    )

    sim_tra_spearman = pair_df_to_similarity_matrix(
        pair_df,
        "spearman_umi_rho_TRA"
    )

    sim_trb_spearman = pair_df_to_similarity_matrix(
        pair_df,
        "spearman_umi_rho_TRB"
    )

    return {
        "group_no": group_no,
        "transform": transform,
        "common_mice": common_mice,

        "TRA_mouse_clone_mat": tra_mat,
        "TRB_mouse_clone_mat": trb_mat,

        "pair_df": pair_df,
        "summary": summary,

        "sim_TRA_pearson": sim_tra_pearson,
        "sim_TRB_pearson": sim_trb_pearson,

        "sim_TRA_spearman": sim_tra_spearman,
        "sim_TRB_spearman": sim_trb_spearman,
    }






def plot_tra_trb_corr_heatmaps(result, corr_type="pearson", save_path=None, show=True):
    group_no = result["group_no"]
    transform = result["transform"]
    if corr_type == "pearson":
        sim_tra = result["sim_TRA_pearson"]
        sim_trb = result["sim_TRB_pearson"]
        title_corr = "Pearson correlation of clonotype UMI counts"
    elif corr_type == "spearman":
        sim_tra = result["sim_TRA_spearman"]
        sim_trb = result["sim_TRB_spearman"]
        title_corr = "Spearman correlation of clonotype UMI counts"
    else:
        raise ValueError("corr_type must be 'pearson' or 'spearman'")

    fig = plt.figure(figsize=(15, 6.5), constrained_layout=True)
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.045])
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
    color_axis = fig.add_subplot(grid[0, 2])
    images = []
    for axis, matrix, chain_label in zip(axes, [sim_tra, sim_trb], ["TRA", "TRB"]):
        image = axis.imshow(
            matrix.values,
            aspect="equal",
            interpolation="none",
            vmin=-1,
            vmax=1,
            cmap="magma",
        )
        images.append(image)
        axis.set_title(f"{chain_label}: {title_corr}")
        axis.set_xticks(
            range(len(matrix.columns)), labels=matrix.columns, rotation=45, ha="right"
        )
        axis.set_yticks(range(len(matrix.index)), labels=matrix.index)
        axis.grid(False)
        axis.tick_params(which="minor", bottom=False, left=False)
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                value = float(matrix.iloc[row_index, column_index])
                if np.isfinite(value):
                    axis.text(
                        column_index,
                        row_index,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color=_adaptive_annotation_color(image, value),
                    )

    colorbar = fig.colorbar(images[-1], cax=color_axis)
    colorbar.set_label("Correlation between mice")
    fig.suptitle(
        f"TRA and TRB | treatment group {group_no} | {STRATUM_LABEL}\n"
        f"Pairwise clonotype UMI correlation; transform={transform}",
        fontsize=14,
    )
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close()



def plot_tra_trb_paircorr_scatter(result, corr_type="pearson", save_path=None, show=True):
    pair_df = result["pair_df"].copy()
    summary = result["summary"].iloc[0]
    group_no = result["group_no"]
    transform = result["transform"]

    if corr_type == "pearson":
        x_col = "pearson_umi_corr_TRA"
        y_col = "pearson_umi_corr_TRB"
        label = "Pearson correlation of UMI counts"
        rho_value = summary["TRA_vs_TRB_spearman_rho_of_pairwise_pearson_corr"]
    elif corr_type == "spearman":
        x_col = "spearman_umi_rho_TRA"
        y_col = "spearman_umi_rho_TRB"
        label = "Spearman correlation of UMI counts"
        rho_value = np.nan
    else:
        raise ValueError("corr_type must be 'pearson' or 'spearman'")

    plot_df = pair_df[[x_col, y_col, "mouse_a", "mouse_b"]].dropna().copy()

    plt.figure(figsize=(8, 8))

    plt.scatter(
        plot_df[x_col],
        plot_df[y_col],
        s=85,
        alpha=0.8
    )

    vals = pd.concat([plot_df[x_col], plot_df[y_col]]).dropna()

    if len(vals) > 0:
        mn = min(-1, vals.min())
        mx = max(1, vals.max())

        plt.plot(
            [mn, mx],
            [mn, mx],
            linestyle=":",
            linewidth=1.5,
            label="TRA = TRB"
        )

        plt.xlim(mn, mx)
        plt.ylim(mn, mx)

    texts = []

    for _, row in plot_df.iterrows():
        txt = plt.text(
            row[x_col],
            row[y_col],
            f"{row['mouse_a']}–{row['mouse_b']}",
            fontsize=8
        )
        texts.append(txt)

    if len(texts) > 0:
        adjust_text(
            texts,
            arrowprops=dict(
                arrowstyle="-",
                color="gray",
                lw=0.5,
                alpha=0.6
            ),
            iter_lim=700
        )

    plt.xlabel(f"TRA: {label}")
    plt.ylabel(f"TRB: {label}")

    if corr_type == "pearson":
        plt.title(
            f"{group_no}: TRA/TRB concordance\n"
            f"{label}, transform={transform}\n"
            f"Spearman rho between TRA/TRB pairwise correlations = {rho_value:.3f}"
        )
    else:
        plt.title(
            f"{group_no}: TRA/TRB concordance\n"
            f"{label}, transform={transform}"
        )

    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()


def collect_boxplot_table_from_corr_results(results_dict, corr_type="pearson"):
    rows = []

    if corr_type == "pearson":
        tra_col = "pearson_umi_corr_TRA"
        trb_col = "pearson_umi_corr_TRB"
        metric_label = "Pearson UMI-count correlation"
    elif corr_type == "spearman":
        tra_col = "spearman_umi_rho_TRA"
        trb_col = "spearman_umi_rho_TRB"
        metric_label = "Spearman UMI-count correlation"
    else:
        raise ValueError("corr_type must be 'pearson' or 'spearman'")

    for group_no, result in results_dict.items():
        pair_df = result["pair_df"].copy()

        for value in pair_df[tra_col].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRA",
                "correlation": value,
                "metric": metric_label,
                "transform": result["transform"]
            })

        for value in pair_df[trb_col].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRB",
                "correlation": value,
                "metric": metric_label,
                "transform": result["transform"]
            })

    return pd.DataFrame(rows)


def plot_group_chain_corr_boxplot(box_df, save_path=None, show=True):
    labels = []
    data = []

    for group_no in sorted(box_df["group_no"].unique()):
        for chain in ["TRA", "TRB"]:
            values = box_df[
                (box_df["group_no"] == group_no) &
                (box_df["chain"] == chain)
            ]["correlation"].dropna().values

            labels.append(f"{group_no}\n{chain}")
            data.append(values)

    plt.figure(figsize=(10, 6))

    plt.boxplot(data, labels=labels)

    plt.axhline(0, linestyle=":", linewidth=1)

    plt.ylabel("Pairwise correlation of UMI counts between mice")
    plt.title("TRA and TRB within-group correlations of clonotype UMI counts")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()






ct_plan = build_ct_pair_plan(GROUPS_TO_RUN)
umi_corr_results = {}
umi_status = []

for group_no, cts in ct_plan.items():
    print("\n" + "=" * 90)
    print(f"Analyzing {group_no}: TRA vs TRB pairwise UMI clonotype correlations")
    try:
        result = compare_tra_trb_umi_correlations_for_group(
            ct_tra=cts["TRA"],
            ct_trb=cts["TRB"],
            metadata=metadata,
            group_no=group_no,
            transform=TRANSFORM,
            min_clones_union=MIN_CLONES_UNION,
        )
    except ValueError as error:
        umi_status.append(
            {"group_no": group_no, "status": "not estimated", "reason": str(error)}
        )
        print(f"Skipping {group_no} UMI correlation: {error}")
        continue

    umi_corr_results[group_no] = result
    umi_status.append({"group_no": group_no, "status": "estimated", "reason": ""})
    display(result["summary"])
    result["summary"].to_csv(
        f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_corr_summary.csv", index=False
    )
    result["pair_df"].to_csv(
        f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_corr_pairwise.csv", index=False
    )
    plot_tra_trb_corr_heatmaps(
        result,
        corr_type="pearson",
        save_path=f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_pearson_heatmaps.png",
        show=SHOW_FIGURES,
    )
    plot_tra_trb_paircorr_scatter(
        result,
        corr_type="pearson",
        save_path=f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_pearson_scatter.png",
        show=SHOW_FIGURES,
    )

pd.DataFrame(umi_status).to_csv(f"{OUT_DIR}/TRA_TRB_clone_UMI_status.csv", index=False)
if umi_corr_results:
    box_pearson = collect_boxplot_table_from_corr_results(
        umi_corr_results, corr_type="pearson"
    )
    box_pearson.to_csv(
        f"{OUT_DIR}/all_groups_TRA_TRB_clone_UMI_pearson_boxplot_table.csv",
        index=False,
    )
    if not box_pearson.empty:
        plot_group_chain_corr_boxplot(
            box_pearson,
            save_path=f"{OUT_DIR}/all_groups_TRA_TRB_clone_UMI_pearson_boxplot.png",
            show=SHOW_FIGURES,
        )

print("\nCOMPLETE")
print(f"Results saved to: {OUT_DIR}")


## Within-group mouse overlap analysis


In [ ]:
import os
import re
import ast
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib_venn import venn2, venn3

plt.rcParams["font.family"] = "DejaVu Sans"






VENN_OUT_DIR = str(RESULT_DIR / "intragroup_mouse_overlap")
os.makedirs(VENN_OUT_DIR, exist_ok=True)

RUN_CHAINS = ["TRA", "TRB"]



RUN_FEATURE_MODES = ["aaV"]      

SHOW_FIGURES = True






def infer_mouse_id_from_sample_id(sample_id):
    
    sample_id = str(sample_id)
    m = re.search(r"(g\d+)_m(\d+)", sample_id)

    if m is None:
        return sample_id

    return f"{m.group(1)}_m{m.group(2)}"


def prepare_metadata_for_mouse_venn(metadata):
    md = metadata.copy()

    if "sample_id" not in md.columns:
        raise ValueError("Metadata does not contain a sample_id column")

    md["sample_id"] = md["sample_id"].astype(str)

    if "mouse_id" not in md.columns:
        md["mouse_id"] = md["sample_id"].apply(infer_mouse_id_from_sample_id)
    else:
        md["mouse_id"] = md["mouse_id"].astype(str)

    return md


def parse_index_value(x):
    
    if isinstance(x, tuple):
        return x

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, tuple):
                return y
        except Exception:
            pass

    return x


def extract_features_from_count_table_index(ct, feature_mode="aaV"):
    """
    feature_mode='aaV':
        feature = clone_id = (cdr3aa, v)

    feature_mode='v':
        feature = V gene
    """
    if isinstance(ct.index, pd.MultiIndex):
        idx_tuples = list(ct.index.to_flat_index())
    else:
        idx_tuples = [parse_index_value(x) for x in ct.index]

    if feature_mode == "aaV":
        return idx_tuples

    if feature_mode == "v":
        v_genes = []

        for x in idx_tuples:
            if not isinstance(x, tuple) or len(x) < 2:
                raise ValueError(
                    "feature_mode='v' requires a count-table index of (cdr3aa, v)"
                )
            v_genes.append(str(x[1]))

        return v_genes

    raise ValueError("feature_mode must be 'aaV' or 'v'")


def build_mouse_feature_sets_from_count_table(
    ct,
    metadata,
    group_no=None,
    feature_mode="aaV"
):
    
    md = prepare_metadata_for_mouse_venn(metadata)

    if group_no is not None and "group_no" in md.columns:
        md = md[md["group_no"] == group_no].copy()

    sample_to_mouse = dict(zip(md["sample_id"], md["mouse_id"]))

    ct_work = ct.copy()
    ct_work.columns = ct_work.columns.astype(str)

    sample_cols = [c for c in ct_work.columns if c in sample_to_mouse]

    
    if len(sample_cols) == 0:
        sample_cols = list(ct_work.columns)
        sample_to_mouse = {
            sample_id: infer_mouse_id_from_sample_id(sample_id)
            for sample_id in sample_cols
        }

    ct_work = ct_work[sample_cols].copy()
    ct_work = ct_work.apply(pd.to_numeric, errors="coerce").fillna(0)

    features = extract_features_from_count_table_index(
        ct_work,
        feature_mode=feature_mode
    )

    if feature_mode == "aaV":
        ct_work.index = pd.Index(features)

    elif feature_mode == "v":
        ct_work["feature_id"] = features
        ct_work = ct_work.groupby("feature_id").sum()

    ct_work.columns = [sample_to_mouse[c] for c in ct_work.columns]

    
    
    mouse_feature_mat = ct_work.T.groupby(level=0).sum().T

    mouse_sets = {
        mouse_id: set(mouse_feature_mat.index[mouse_feature_mat[mouse_id] > 0])
        for mouse_id in mouse_feature_mat.columns
    }

    
    mouse_sets = {
        mouse_id: feature_set
        for mouse_id, feature_set in mouse_sets.items()
        if len(feature_set) > 0
    }

    return mouse_sets, mouse_feature_mat


def save_mouse_sets_summary(mouse_sets, save_path):
    rows = []

    for mouse_id, feature_set in mouse_sets.items():
        rows.append({
            "mouse_id": mouse_id,
            "n_features": len(feature_set)
        })

    summary = pd.DataFrame(rows, columns=["mouse_id", "n_features"]).sort_values("mouse_id").reset_index(drop=True)
    summary.to_csv(save_path, index=False)
    return summary


def set_venn_style(v, alpha=0.55):
    
    if v is None:
        return

    for patch in v.patches:
        if patch is not None:
            patch.set_alpha(alpha)
            patch.set_edgecolor("black")
            patch.set_linewidth(1.2)

    for label in v.set_labels:
        if label is not None:
            label.set_fontsize(10)
            label.set_fontweight("bold")

    for label in v.subset_labels:
        if label is not None:
            label.set_fontsize(9)
            label.set_bbox(
                dict(
                    boxstyle="round,pad=0.15",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.75
                )
            )


def plot_mouse_venn_panel(
    mouse_sets,
    group_label,
    chain_label,
    feature_mode="aaV",
    save_path=None,
    show=True
):
    
    mouse_ids = sorted(mouse_sets.keys())

    if feature_mode == "aaV":
        feature_label = "aaV clonotypes (CDR3aa, V)"
    elif feature_mode == "v":
        feature_label = "V segments"
    else:
        feature_label = feature_mode

    if len(mouse_ids) < 2:
        print(f"{chain_label}, {group_label}, {feature_mode}: fewer than two non-empty mice")
        return None

    
    
    
    if len(mouse_ids) == 2:
        a, b = mouse_ids

        fig, ax = plt.subplots(figsize=(7, 7))

        v = venn2(
            [mouse_sets[a], mouse_sets[b]],
            set_labels=(a, b),
            ax=ax
        )
        set_venn_style(v)

        ax.set_title(
            f"{chain_label}, {group_label}: mouse overlap for {feature_label}",
            fontsize=12
        )

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close()

        return fig

    
    
    
    if len(mouse_ids) == 3:
        a, b, c = mouse_ids

        fig, ax = plt.subplots(figsize=(8, 8))

        v = venn3(
            [mouse_sets[a], mouse_sets[b], mouse_sets[c]],
            set_labels=(a, b, c),
            ax=ax
        )
        set_venn_style(v)

        ax.set_title(
            f"{chain_label}, {group_label}: mouse overlap for {feature_label}",
            fontsize=12
        )

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close()

        return fig

    
    
    
    combos = list(itertools.combinations(mouse_ids, 3))

    n_panels = len(combos)
    ncols = 2
    nrows = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(13, max(6, 5.3 * nrows))
    )

    axes = np.array(axes).reshape(-1)

    for ax, (a, b, c) in zip(axes, combos):
        v = venn3(
            [mouse_sets[a], mouse_sets[b], mouse_sets[c]],
            set_labels=(a, b, c),
            ax=ax
        )
        set_venn_style(v, alpha=0.50)
        ax.set_title(f"{a}, {b}, {c}", fontsize=11)

    for ax in axes[len(combos):]:
        ax.axis("off")

    fig.suptitle(
        f"{chain_label}, {group_label}: all three-mouse Venn diagrams for {feature_label}",
        fontsize=14,
        y=1.005
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return fig


def run_one_mouse_venn_analysis(
    ct,
    metadata,
    group_no,
    group_label,
    chain_label,
    feature_mode,
    out_dir=VENN_OUT_DIR,
    show=True
):
    mouse_sets, mouse_feature_mat = build_mouse_feature_sets_from_count_table(
        ct=ct,
        metadata=metadata,
        group_no=group_no,
        feature_mode=feature_mode
    )

    safe_group_label = group_label.replace("/", "_")
    safe_feature_mode = feature_mode.replace("/", "_")

    summary_path = os.path.join(
        out_dir,
        f"{chain_label}_{safe_group_label}_{safe_feature_mode}_mouse_feature_counts.csv"
    )

    matrix_path = os.path.join(
        out_dir,
        f"{chain_label}_{safe_group_label}_{safe_feature_mode}_mouse_feature_matrix.csv"
    )

    fig_path = os.path.join(
        out_dir,
        f"{chain_label}_{safe_group_label}_{safe_feature_mode}_mouse_venn_panel.png"
    )

    summary_df = save_mouse_sets_summary(mouse_sets, summary_path)
    mouse_feature_mat.to_csv(matrix_path)

    print("\n" + "=" * 80)
    print(f"{chain_label}, {group_label}, {feature_mode}")
    print(f"Mice: {len(mouse_sets)}")
    display(summary_df)

    plot_mouse_venn_panel(
        mouse_sets=mouse_sets,
        group_label=group_label,
        chain_label=chain_label,
        feature_mode=feature_mode,
        save_path=fig_path,
        show=show
    )

    return {
        "mouse_sets": mouse_sets,
        "mouse_feature_mat": mouse_feature_mat,
        "summary_df": summary_df,
        "summary_path": summary_path,
        "matrix_path": matrix_path,
        "fig_path": fig_path
    }


def get_existing_count_table(var_name):
    
    return globals().get(var_name, None)


def build_count_table_plan():
    
    plan = []

    for chain in RUN_CHAINS:
        chain_low = chain.lower()

        
        for group_no in ["g1", "g5", "g6"]:
            var_name = f"ct_{group_no}_aaV_{chain_low}"
            ct = get_existing_count_table(var_name)

            if ct is not None:
                plan.append({
                    "group_no": group_no,
                    "group_label": group_no,
                    "chain_label": chain,
                    "ct_name": var_name,
                    "ct": ct
                })
            else:
                print(f"Skipping: missing variable {var_name}")

        
        for g6_part in ["g6_auto", "g6_allo"]:
            var_name = f"ct_{g6_part}_aaV_{chain_low}"
            ct = get_existing_count_table(var_name)

            if ct is not None:
                plan.append({
                    "group_no": "g6",
                    "group_label": g6_part,
                    "chain_label": chain,
                    "ct_name": var_name,
                    "ct": ct
                })
            else:
                print(f"Optional step skipped: missing variable {var_name}")

    return plan






venn_results = {}

ct_plan = build_count_table_plan()

for item in ct_plan:
    for feature_mode in RUN_FEATURE_MODES:
        key = (
            item["chain_label"],
            item["group_label"],
            feature_mode
        )

        venn_results[key] = run_one_mouse_venn_analysis(
            ct=item["ct"],
            metadata=metadata,
            group_no=item["group_no"],
            group_label=item["group_label"],
            chain_label=item["chain_label"],
            feature_mode=feature_mode,
            out_dir=VENN_OUT_DIR,
            show=SHOW_FIGURES
        )

print("\nCOMPLETE")
print(f"All figures and tables were saved to: {VENN_OUT_DIR}")

## Differential clonotype abundance and TRAV prioritization

The prespecified inferential contrast compares g1 with the pooled g5+g6 allogeneic
reference. edgeR retains independent sample or mouse-level analysis units. Fisher's
exact test provides a separate pooled-count sensitivity analysis and is not treated as
an independent biological replicate model.


In [ ]:
ACTIVE_CHAIN = "TRA"
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

clonosets_tra_for_diffexp = pd.concat(
    [clonosets_tra_g1, clonosets_tra_g2, clonosets_tra_g5, clonosets_tra_g6],
    ignore_index=True,
)
ct_tra_aaV = intersections.count_table(
    clonosets_tra_for_diffexp,
    overlap_type="aaV",
    mismatches=0,
).fillna(0)
ct_tra_aaV = ct_tra_aaV.apply(pd.to_numeric, errors="coerce").fillna(0).round().astype(int)

sample_metadata = (
    metadata[metadata["sample_id"].isin(ct_tra_aaV.columns)]
    .drop_duplicates("sample_id")
    .set_index("sample_id")
    .loc[ct_tra_aaV.columns]
    .copy()
)
sample_metadata["analysis_unit"] = sample_metadata.index.astype(str)
if POOL_WITHIN_MOUSE:
    sample_metadata["analysis_unit"] = sample_metadata["mouse_id"].astype(str)
    unit_by_sample = sample_metadata["analysis_unit"].to_dict()
    ct_tra_aaV.columns = [unit_by_sample[str(column)] for column in ct_tra_aaV.columns]
    ct_tra_aaV = ct_tra_aaV.T.groupby(level=0, observed=True).sum().T.astype(int)
    sample_metadata = (
        sample_metadata.reset_index()
        .drop_duplicates("analysis_unit")
        .set_index("analysis_unit")
        .loc[ct_tra_aaV.columns]
    )
else:
    sample_metadata = sample_metadata.set_index("analysis_unit", drop=True)

sample_metadata["contrast_group"] = sample_metadata["group_no"].replace(
    {"g5": "allogeneic", "g6": "allogeneic", "g1": "g1"}
)
count_table_path = RESULT_DIR / "tra_aav_count_matrix.csv"
ct_tra_aaV.to_csv(count_table_path)
display(sample_metadata.groupby(["group_no", "contrast_group"], observed=True).size())
print(f"Differential count matrix: {ct_tra_aaV.shape}")


In [ ]:
fisher_samples_g1 = sample_metadata.index[sample_metadata["group_no"].eq("g1")].tolist()
fisher_samples_ctrl = sample_metadata.index[
    sample_metadata["group_no"].isin(["g5", "g6"])
].tolist()

g1_counts = ct_tra_aaV[fisher_samples_g1].sum(axis=1).astype(float)
control_counts = ct_tra_aaV[fisher_samples_ctrl].sum(axis=1).astype(float)
total_g1 = float(g1_counts.sum())
total_control = float(control_counts.sum())

fisher_rows = []
for feature_id, g1_count, control_count in zip(
    ct_tra_aaV.index, g1_counts, control_counts
):
    table = [
        [g1_count, control_count],
        [max(total_g1 - g1_count, 0), max(total_control - control_count, 0)],
    ]
    odds_ratio, p_value = fisher_exact(table, alternative="two-sided")
    fisher_rows.append(
        {
            "feature_id": feature_id,
            "fisher_g1_count": g1_count,
            "fisher_allogeneic_count": control_count,
            "Fisher_log2FC_g1": np.log2((g1_count + 1) / (control_count + 1)),
            "Fisher_odds_ratio": odds_ratio,
            "Fisher_p_g1": p_value,
        }
    )

fisher_results = pd.DataFrame(
    fisher_rows,
    columns=[
        "feature_id",
        "fisher_g1_count",
        "fisher_allogeneic_count",
        "Fisher_log2FC_g1",
        "Fisher_odds_ratio",
        "Fisher_p_g1",
    ],
)
if len(fisher_results):
    fisher_results["Fisher_FDR_g1"] = multipletests(
        fisher_results["Fisher_p_g1"], method="fdr_bh"
    )[1]
else:
    fisher_results["Fisher_FDR_g1"] = pd.Series(dtype=float)
fisher_results.to_csv(RESULT_DIR / "fisher_aav_results.csv", index=False)
display(fisher_results.sort_values("Fisher_FDR_g1").head(20))


In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

edgeR_columns = ["logFC", "logCPM", "F", "PValue", "FDR", "feature_id"]
edgeR_res_g1 = pd.DataFrame(columns=edgeR_columns)
EDGER_STATUS = "not fitted"

edgeR_units = sample_metadata.index[
    sample_metadata["contrast_group"].isin(["g1", "allogeneic"])
].tolist()
countData_g1 = ct_tra_aaV[edgeR_units].copy()
coldata_g1 = sample_metadata.loc[edgeR_units, ["contrast_group"]].copy()
replicate_counts = coldata_g1["contrast_group"].value_counts()

if replicate_counts.get("g1", 0) < 2 or replicate_counts.get("allogeneic", 0) < 2:
    EDGER_STATUS = (
        "edgeR was not fitted because at least two independent analysis units "
        "were required in both contrast groups."
    )
else:
    feature_lookup = pd.DataFrame(
        {
            "feature_key": [f"feature_{index:07d}" for index in range(len(countData_g1))],
            "feature_id": list(countData_g1.index),
        }
    )
    countData_g1.index = feature_lookup["feature_key"]
    with (ro.default_converter + pandas2ri.converter).context():
        converter = ro.conversion.get_conversion()
        ro.globalenv["countData"] = converter.py2rpy(countData_g1)
        ro.globalenv["coldata"] = converter.py2rpy(coldata_g1.reset_index(drop=True))

    ro.r(
        """
        suppressPackageStartupMessages(library(edgeR))
        countData <- as.matrix(countData)
        storage.mode(countData) <- "integer"
        group <- factor(coldata$contrast_group, levels=c("allogeneic", "g1"))
        design <- model.matrix(~ group)
        y <- DGEList(counts=countData, group=group)
        y <- calcNormFactors(y)
        keep <- filterByExpr(y, design=design)
        if (any(keep)) {
            y <- y[keep, , keep.lib.sizes=FALSE]
            y <- estimateDisp(y, design)
            fit <- glmQLFit(y, design)
            test <- glmQLFTest(fit, coef=2)
            edgeR_table <- topTags(test, n=Inf)$table
        } else {
            edgeR_table <- data.frame()
        }
        """
    )
    with (ro.default_converter + pandas2ri.converter).context():
        converted = ro.conversion.get_conversion().rpy2py(ro.globalenv["edgeR_table"])
    edgeR_res_g1 = pd.DataFrame(converted)
    if not edgeR_res_g1.empty:
        edgeR_res_g1["feature_key"] = edgeR_res_g1.index.astype(str)
        edgeR_res_g1 = edgeR_res_g1.merge(feature_lookup, on="feature_key", how="left")
        edgeR_res_g1 = edgeR_res_g1.drop(columns="feature_key")
        EDGER_STATUS = "edgeR quasi-likelihood model fitted successfully."
    else:
        edgeR_res_g1 = pd.DataFrame(columns=edgeR_columns)
        EDGER_STATUS = "No aaV clonotype passed edgeR expression filtering."

edgeR_res_g1.to_csv(RESULT_DIR / "edger_aav_results.csv", index=False)
print(EDGER_STATUS)
display(edgeR_res_g1.head())


In [ ]:
def _empty_axis(ax, message):
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", transform=ax.transAxes, wrap=True)


edger_plot = edgeR_res_g1.copy()
if not edger_plot.empty:
    edger_plot["minus_log10_FDR"] = -np.log10(
        pd.to_numeric(edger_plot["FDR"], errors="coerce").clip(lower=1e-300)
    )
    edger_plot["significant"] = edger_plot["FDR"].lt(0.05)

fig, ax = plt.subplots(figsize=(9, 7))
if edger_plot.empty:
    _empty_axis(ax, EDGER_STATUS)
else:
    sns.scatterplot(
        data=edger_plot, x="logFC", y="minus_log10_FDR", hue="significant",
        palette={True: "#B2182B", False: "#9E9E9E"}, alpha=0.7, s=35,
        linewidth=0, ax=ax,
    )
    ax.axvline(1, linestyle="--", color="black")
    ax.axvline(-1, linestyle="--", color="black")
    ax.axhline(-np.log10(0.05), linestyle="--", color="black")
    ax.set_xlabel("edgeR log2 fold change: g1 / g5+g6")
    ax.set_ylabel("-log10 FDR")
ax.set_title(f"edgeR differential aaV abundance: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
if edger_plot.empty:
    _empty_axis(ax, EDGER_STATUS)
else:
    sns.scatterplot(
        data=edger_plot, x="logFC", y="minus_log10_FDR", hue="significant",
        palette={True: "#B2182B", False: "#9E9E9E"}, alpha=0.7, s=35,
        linewidth=0, legend=False, ax=ax,
    )
    for row in edger_plot.sort_values(["FDR", "logFC"], ascending=[True, False]).head(15).itertuples():
        ax.annotate(str(row.feature_id), (row.logFC, row.minus_log10_FDR), fontsize=7)
    ax.set_xlabel("edgeR log2 fold change: g1 / g5+g6")
    ax.set_ylabel("-log10 FDR")
ax.set_title(f"edgeR leading aaV clonotypes: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
if edger_plot.empty:
    _empty_axis(ax, EDGER_STATUS)
else:
    sns.scatterplot(
        data=edger_plot, x="logCPM", y="logFC", hue="significant",
        palette={True: "#B2182B", False: "#9E9E9E"}, alpha=0.7, s=35,
        linewidth=0, ax=ax,
    )
    ax.axhline(1, linestyle="--", color="black")
    ax.axhline(-1, linestyle="--", color="black")
    ax.set_xlabel("Mean clonotype abundance, logCPM")
    ax.set_ylabel("edgeR log2 fold change: g1 / g5+g6")
ax.set_title(f"edgeR MA plot: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
if edger_plot.empty:
    _empty_axis(ax, EDGER_STATUS)
else:
    sns.scatterplot(
        data=edger_plot, x="logCPM", y="logFC", hue="significant",
        palette={True: "#B2182B", False: "#9E9E9E"}, alpha=0.7, s=35,
        linewidth=0, ax=ax,
    )
    ax.axhline(1, linestyle="--", color="black")
    ax.axhline(-1, linestyle="--", color="black")
    for row in edger_plot.sort_values(["FDR", "logFC"], ascending=[True, False]).head(15).itertuples():
        ax.annotate(str(row.feature_id), (row.logCPM, row.logFC), fontsize=7)
    ax.set_xlabel("Mean clonotype abundance, logCPM")
    ax.set_ylabel("edgeR log2 fold change: g1 / g5+g6")
ax.set_title(f"edgeR MA plot with leading clonotypes: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()


In [ ]:
def _feature_to_cdr3_v(feature):
    if isinstance(feature, tuple) and len(feature) >= 2:
        return str(feature[0]), str(feature[1])
    try:
        parsed = ast.literal_eval(str(feature))
    except (ValueError, SyntaxError):
        parsed = None
    if isinstance(parsed, tuple) and len(parsed) >= 2:
        return str(parsed[0]), str(parsed[1])
    text = str(feature)
    match = re.search(r"(TRAV[0-9A-Za-z./-]+)", text)
    return text, match.group(1) if match else "unresolved"


if not edger_plot.empty:
    parsed_features = edger_plot["feature_id"].map(_feature_to_cdr3_v)
    edger_plot["cdr3aa"] = parsed_features.map(lambda value: value[0])
    edger_plot["v_gene"] = parsed_features.map(lambda value: value[1])
else:
    edger_plot["cdr3aa"] = pd.Series(dtype=str)
    edger_plot["v_gene"] = pd.Series(dtype=str)

if not edger_plot.empty:
    edgeR_trav = edger_plot.groupby("v_gene", as_index=False).agg(
        n_edger_tested=("feature_id", "count"),
        min_edger_fdr=("FDR", "min"),
    )
    edgeR_significant = (
        edger_plot[edger_plot["FDR"].lt(0.05)]
        .groupby("v_gene", as_index=False)
        .agg(
            n_edger_significant=("feature_id", "count"),
            mean_edger_log2fc=("logFC", "mean"),
            max_edger_log2fc=("logFC", "max"),
        )
    )
    edgeR_trav = edgeR_trav.merge(edgeR_significant, on="v_gene", how="left")
else:
    edgeR_trav = pd.DataFrame(
        columns=[
            "v_gene", "n_edger_tested", "n_edger_significant",
            "mean_edger_log2fc", "max_edger_log2fc", "min_edger_fdr",
        ]
    )

exclusive_tra = pd.DataFrame(sorted(g1_remaining_aaV_tra), columns=["cdr3aa", "v_gene"])
exclusive_trav = (
    exclusive_tra.groupby("v_gene", as_index=False)
    .size()
    .rename(columns={"size": "n_g1_exclusive_aav"})
)

fisher_parsed = fisher_results.copy()
if not fisher_parsed.empty:
    parsed_fisher = fisher_parsed["feature_id"].map(_feature_to_cdr3_v)
    fisher_parsed["v_gene"] = parsed_fisher.map(lambda value: value[1])
    fisher_trav = fisher_parsed.groupby("v_gene", as_index=False).agg(
        min_fisher_fdr=("Fisher_FDR_g1", "min"),
    )
    fisher_significant = (
        fisher_parsed[fisher_parsed["Fisher_FDR_g1"].lt(0.05)]
        .groupby("v_gene", as_index=False)
        .agg(
            n_fisher_significant=("feature_id", "count"),
            mean_fisher_log2fc=("Fisher_log2FC_g1", "mean"),
        )
    )
    fisher_trav = fisher_trav.merge(fisher_significant, on="v_gene", how="left")
else:
    fisher_trav = pd.DataFrame(
        columns=["v_gene", "n_fisher_significant", "mean_fisher_log2fc", "min_fisher_fdr"]
    )

trav_ranking = exclusive_trav.merge(edgeR_trav, on="v_gene", how="outer").merge(
    fisher_trav, on="v_gene", how="outer"
)
count_columns = [
    "n_g1_exclusive_aav", "n_edger_tested", "n_edger_significant", "n_fisher_significant"
]
for column in count_columns:
    trav_ranking[column] = pd.to_numeric(trav_ranking[column], errors="coerce").fillna(0).astype(int)

trav_ranking["edgeR_g1_enriched"] = (
    trav_ranking["n_edger_significant"].gt(0)
    & trav_ranking["mean_edger_log2fc"].gt(0)
)
trav_ranking["fisher_g1_enriched"] = (
    trav_ranking["n_fisher_significant"].gt(0)
    & trav_ranking["mean_fisher_log2fc"].gt(0)
)
trav_ranking["evidence_class"] = np.select(
    [
        trav_ranking["edgeR_g1_enriched"] & trav_ranking["fisher_g1_enriched"],
        trav_ranking["edgeR_g1_enriched"],
        trav_ranking["fisher_g1_enriched"],
        trav_ranking["n_g1_exclusive_aav"].gt(0),
    ],
    [
        "edgeR and Fisher concordant",
        "edgeR-supported",
        "Fisher-supported",
        "exact-set descriptive",
    ],
    default="tested without positive enrichment evidence",
)
trav_ranking["evidence_score"] = (
    4 * trav_ranking["edgeR_g1_enriched"].astype(int)
    + 2 * trav_ranking["fisher_g1_enriched"].astype(int)
    + np.log1p(trav_ranking["n_g1_exclusive_aav"])
)
trav_ranking["stratum"] = STRATUM
trav_ranking["stratum_label"] = STRATUM_LABEL
trav_ranking = trav_ranking.sort_values(
    ["evidence_score", "n_edger_significant", "n_g1_exclusive_aav", "v_gene"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
trav_ranking["stratum_rank"] = np.arange(1, len(trav_ranking) + 1)
trav_ranking.to_csv(RESULT_DIR / "trav_ranking.csv", index=False)

distinctive_trav = trav_ranking[
    trav_ranking["evidence_class"].ne("tested without positive enrichment evidence")
].head(20).copy()
if distinctive_trav.empty:
    distinctive_trav = trav_ranking.head(10).copy()
    distinctive_trav["evidence_class"] = "descriptive ranking; no positive inferential evidence"
distinctive_trav.to_csv(RESULT_DIR / "distinctive_trav.csv", index=False)
display(distinctive_trav)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
top_effect = trav_ranking[
    trav_ranking["n_edger_significant"].gt(0)
    & trav_ranking["mean_edger_log2fc"].notna()
].head(15).sort_values("mean_edger_log2fc")
if top_effect.empty:
    _empty_axis(ax, "No TRAV segment contained an edgeR-significant aaV clonotype.")
else:
    ax.barh(top_effect["v_gene"], top_effect["mean_edger_log2fc"], color="#4C78A8")
    ax.set_xlabel("Mean edgeR log2 fold change among significant aaV clonotypes")
    ax.set_ylabel("TRAV segment")
ax.set_title(f"TRAV segments ranked by edgeR effect: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
top_count = trav_ranking[trav_ranking["n_edger_significant"].gt(0)].head(15)
top_count = top_count.sort_values("n_edger_significant")
if top_count.empty:
    _empty_axis(ax, "No TRAV segment contained an edgeR-significant aaV clonotype.")
else:
    ax.barh(top_count["v_gene"], top_count["n_edger_significant"], color="#7A5195")
    ax.set_xlabel("edgeR-significant aaV clonotypes")
    ax.set_ylabel("TRAV segment")
ax.set_title(f"TRAV segments ranked by significant-clonotype count: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()

comparison_g1 = fisher_results.merge(edgeR_res_g1, on="feature_id", how="inner")
comparison_g1.to_csv(RESULT_DIR / "fisher_edger_feature_comparison.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 8))
plot_df = comparison_g1.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["Fisher_log2FC_g1", "logFC"]
)
if plot_df.empty:
    _empty_axis(ax, "No aaV clonotype was jointly available from Fisher and edgeR.")
else:
    ax.scatter(plot_df["Fisher_log2FC_g1"], plot_df["logFC"], s=20, alpha=0.6)
    lower = min(plot_df["Fisher_log2FC_g1"].min(), plot_df["logFC"].min())
    upper = max(plot_df["Fisher_log2FC_g1"].max(), plot_df["logFC"].max())
    ax.plot([lower, upper], [lower, upper], linestyle="--", color="black", linewidth=1)
    ax.set_xlabel("Fisher pooled-count log2 ratio: g1 / g5+g6")
    ax.set_ylabel("edgeR log2 fold change: g1 / g5+g6")
ax.set_title(f"Fisher and edgeR effect concordance: {STRATUM_LABEL}")
plt.tight_layout()
plt.show()


In [ ]:
top_rows = distinctive_trav.head(5)
candidate_text = (
    "; ".join(
        f"{row.v_gene} [{row.evidence_class}]" for row in top_rows.itertuples()
    )
    if len(top_rows)
    else "none"
)
conclusion_lines = [
    f"# Stratum-specific conclusion: {STRATUM_LABEL}",
    "",
    f"- Selected samples and analysis units are documented in `{RESULT_DIR.name}/selected_samples_by_chain_and_group.csv`.",
    f"- edgeR status: {EDGER_STATUS}",
    f"- Highest-ranked TRAV segments: {candidate_text}.",
    "- Evidence classes distinguish concordant inference, single-method support, and exact-set descriptive ranking.",
    "- Absence of FDR-supported features is retained as a null result and is not converted into evidence of equivalence.",
]
(RESULT_DIR / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")

print("\n".join(conclusion_lines))


In [ ]:
_persist_open_figures(origin="final-cell")

manifest = pd.DataFrame(PLOT_RECORDS)
if manifest.empty:
    raise RuntimeError("No figure was persisted; the notebook output is incomplete.")
manifest["png"] = manifest["png"].map(lambda value: str(Path(value).relative_to(REPOSITORY_ROOT)))
manifest["pdf"] = manifest["pdf"].map(lambda value: str(Path(value).relative_to(REPOSITORY_ROOT)))
manifest.to_csv(RESULT_DIR / "figure_manifest.csv", index=False)

missing_figure_files = []
for record in PLOT_RECORDS:
    for key in ("png", "pdf"):
        if not Path(record[key]).is_file():
            missing_figure_files.append(record[key])
if missing_figure_files:
    raise RuntimeError(f"Missing persisted figure files: {missing_figure_files[:10]}")

run_summary = {
    "stratum": STRATUM,
    "stratum_label": STRATUM_LABEL,
    "pool_within_mouse": POOL_WITHIN_MOUSE,
    "n_selected_samples": int(metadata["sample_id"].nunique()),
    "n_selected_mice": int(metadata["mouse_id"].nunique()),
    "n_figures": int(len(manifest)),
    "n_ranked_trav": int(len(trav_ranking)),
    "n_distinctive_trav": int(len(distinctive_trav)),
    "edger_status": EDGER_STATUS,
}
(RESULT_DIR / "run_summary.json").write_text(
    json.dumps(run_summary, indent=2) + "\n", encoding="utf-8"
)

if os.environ.get("MICE_TCR_FINALIZE_SUMMARY", "0") == "1":
    summary_tables = []
    for stratum_name in STRATUM_CONFIG:
        table_path = RESULT_ROOT / stratum_name / "distinctive_trav.csv"
        if table_path.is_file():
            summary_tables.append(pd.read_csv(table_path))
    if summary_tables:
        eight_stratum_summary = pd.concat(summary_tables, ignore_index=True)
        if eight_stratum_summary.empty:
            raise RuntimeError(
                "The consolidated TRAV table is empty; inspect the eight stratum "
                "executions before interpreting a biological null result."
            )
        eight_stratum_summary.to_csv(
            RESULT_ROOT / "eight_stratum_distinctive_trav_summary.csv", index=False
        )
        display(eight_stratum_summary)

        cross_dir = FIGURE_ROOT / "cross_stratum"
        cross_dir.mkdir(parents=True, exist_ok=True)
        heatmap_source = eight_stratum_summary.copy()
        heatmap_source["display_score"] = heatmap_source["evidence_score"].fillna(0)
        leading_genes = (
            heatmap_source.groupby("v_gene")["display_score"]
            .max()
            .sort_values(ascending=False)
            .head(25)
            .index
        )
        heatmap_table = (
            heatmap_source[heatmap_source["v_gene"].isin(leading_genes)]
            .pivot_table(index="v_gene", columns="stratum", values="display_score", aggfunc="max", fill_value=0)
            .reindex(columns=list(STRATUM_CONFIG))
        )
        fig, ax = plt.subplots(figsize=(13, max(6, 0.35 * len(heatmap_table))))
        sns.heatmap(heatmap_table, cmap="viridis", linewidths=0.2, ax=ax)
        ax.set_xlabel("Biological stratum")
        ax.set_ylabel("TRAV segment")
        ax.set_title("Distinctive TRAV evidence across eight biological strata")
        plt.tight_layout()
        _save_figure_pair(
            fig,
            cross_dir / "distinctive_trav_across_eight_strata.png",
            origin="cross-stratum",
        )
        plt.show()

print(f"Persisted figures: {len(PLOT_RECORDS)}")
print(f"Executed-notebook result set completed for {STRATUM}.")
